# 🧠 Simulador Intensivo — Clase 03: Traduce tu Asignatura a Datos

**Módulo 01 — Pensar con Datos | FACES-UCAB | Ulises González**

Este notebook es un simulador interactivo donde vas a:
1. **Investigar un caso real** de deserción estudiantil tomando decisiones paso a paso
2. **Detectar manipulaciones** en visualizaciones de datos económicos
3. **Analizar datos reales de Venezuela** descargados en vivo desde APIs públicas
4. **Diseñar tu propio proyecto de datos** con feedback de IA en tiempo real
5. **Resolver un quiz integrador** que pone a prueba todo el módulo

Cada respuesta que des será evaluada por un modelo de lenguaje corriendo en una **NVIDIA DGX Spark** 🚀

**Instrucción:** Ejecuta cada celda en orden, una por una. Lee el resultado antes de avanzar.

---

> **¿Por qué?** Colab no trae todas las librerías preinstaladas. Instalamos `openai` (cliente para conectar con el LLM), `plotly` (gráficos interactivos) e `ipywidgets` (controles como botones y dropdowns). Sin este paso, las celdas siguientes fallarían al importar.

### 🐍 Python en esta celda

**`!pip install`** — El signo `!` ejecuta un comando del sistema operativo (no es Python puro). `pip` es el instalador de paquetes de Python. La flag `-q` significa *quiet*: instala sin mostrar todo el detalle.

**`print()`** — La función más básica de Python: muestra texto en pantalla. Todo lo que pongas entre comillas dentro de `print()` se imprime como texto.

In [ ]:
#@title 1. Instalar dependencias
!pip install -q openai plotly ipywidgets
print('✅ Dependencias instaladas')

✅ Dependencias instaladas


> **¿Por qué?** Separar instalación de importación es buena práctica: instalar descarga paquetes (solo se hace una vez), importar los carga en memoria para usarlos. Cada librería cumple un rol: `pandas` para tablas, `numpy` para cálculos, `plotly` para gráficos interactivos, `widgets` para la interfaz del simulador, y `requests` para descargar datos de APIs.

### 🐍 Python en esta celda

**`import ... as ...`** — Carga una librería y le da un alias corto. `import pandas as pd` significa: "carga pandas y déjame llamarlo `pd`". Es una convención universal — si ves `pd` en cualquier código, sabes que es pandas.

**`from ... import ...`** — Importa solo funciones específicas de una librería, sin cargar todo. `from IPython.display import HTML` trae solo la función `HTML`, no todo IPython.

**Librerías que usaremos:**
| Alias | Librería | Para qué sirve |
|-------|----------|----------------|
| `pd` | pandas | Tablas y datos estructurados |
| `np` | numpy | Cálculos numéricos y arrays |
| `go` | plotly.graph_objects | Gráficos interactivos personalizados |
| `px` | plotly.express | Gráficos rápidos con una línea |
| `widgets` | ipywidgets | Botones, dropdowns, campos de texto |
| `requests` | requests | Descargar datos de internet (APIs) |

In [ ]:
#@title 2. Importar librerías
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output, Markdown
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import json
import re
import time
import requests
print('✅ Librerías cargadas')

✅ Librerías cargadas


> **¿Por qué?** Este simulador usa un modelo de lenguaje (IA) para darte feedback personalizado en tiempo real. En vez de respuestas pre-grabadas, la IA lee lo que escribes y te da retroalimentación adaptada. Configuramos aquí la conexión para que todas las celdas posteriores puedan «preguntarle» a la IA.

### 🐍 Python en esta celda

**`def nombre_funcion(parametros):`** — Define una función reutilizable. Todo lo indentado debajo pertenece a la función. Se ejecuta cuando la *llamas*: `nombre_funcion(valor)`.

**`try: ... except: ...`** — Manejo de errores. El código dentro de `try` se intenta ejecutar. Si falla, en vez de detenerse, salta al bloque `except` y ejecuta un plan B. Útil para conexiones de red que pueden fallar.

**`client = OpenAI(...)`** — Crea un *objeto* que representa la conexión con la IA. Piensa en `client` como un teléfono ya marcado: cada vez que quieras hablar con la IA, usas ese objeto.

**`return`** — Devuelve un valor desde una función. Sin `return`, la función hace cosas pero no te da un resultado que puedas guardar.

In [ ]:
#@title 3. Conectar al LLM (DGX Spark)
from openai import OpenAI

LLM_URL = "https://ollama.rizo.ma/v1"
MODEL_NAME = "qwen3:14b"
client = OpenAI(base_url=LLM_URL, api_key="not-needed")

def ask_llm(system_prompt, user_msg, max_tokens=1024):
    """Envía una consulta al LLM y retorna la respuesta limpia."""
    try:
        r = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_msg}
            ],
            max_tokens=max_tokens,
            temperature=0.7
        )
        text = r.choices[0].message.content
        text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
        return text
    except Exception as e:
        return f"⚠️ Error conectando al LLM: {e}\n\nVerifica que ollama.rizo.ma esté accesible."

print('Verificando conexión al LLM...')
test = ask_llm("Responde solo 'OK'", "Test de conexión")
if "Error" in test:
    print(f'❌ {test}')
else:
    print(f'✅ Conectado a DGX Spark — Qwen3-Coder-Next listo')

Verificando conexión al LLM...
✅ Conectado a DGX Spark — Qwen3-Coder-Next listo


> **¿Por qué?** El puntaje no es para calificarte — es para que tú mismo identifiques tus áreas fuertes y débiles. Al acumular puntos por módulo, al final puedes ver un radar de competencias que te dice dónde enfocarte. Esto se llama *evaluación formativa*: mide para aprender, no para aprobar.

### 🐍 Python en esta celda

**Diccionario (`{}`)** — Una estructura que guarda datos como pares `clave: valor`. `score = {"total": 0, "max": 0}` es como una ficha con campos: puedes leer `score["total"]` o modificarlo con `score["total"] += 5`.

**`+=`** — Atajo para sumar y guardar. `x += 3` es lo mismo que `x = x + 3`.

**`if ... not in ...`** — Verifica si algo NO está en una colección. `if modulo not in score["modulo"]` pregunta: "¿este módulo ya tiene registro?" Si no, lo crea.

**`f"...{variable}..."`** — *f-string*: permite insertar valores de variables dentro de texto. `f"Tienes {puntos} puntos"` reemplaza `{puntos}` por su valor real.

In [ ]:
#@title 4. Sistema de puntaje
score = {"total": 0, "max": 0, "modulo": {}}

def add_score(modulo, puntos, maximo):
    score["total"] += puntos
    score["max"] += maximo
    if modulo not in score["modulo"]:
        score["modulo"][modulo] = {"total": 0, "max": 0}
    score["modulo"][modulo]["total"] += puntos
    score["modulo"][modulo]["max"] += maximo

def show_score():
    pct = (score['total'] / score['max'] * 100) if score['max'] > 0 else 0
    color = '#4CAF50' if pct >= 70 else '#FF9800' if pct >= 50 else '#f44336'
    display(HTML(f"""
    <div style='background:{color};color:white;padding:10px 20px;border-radius:8px;
                display:inline-block;font-size:16px;font-weight:bold;'>
        Puntaje acumulado: {score['total']}/{score['max']} ({pct:.0f}%)
    </div>"""))

print('✅ Sistema de puntaje listo')
print('📊 Simulador configurado. ¡Comencemos!')

✅ Sistema de puntaje listo
📊 Simulador configurado. ¡Comencemos!


---
## 🔌 Módulo 0: Recap Relámpago

> **¿Por qué empezamos con un recap?** Activar conocimiento previo antes de aprender algo nuevo mejora la retención (efecto de *priming* cognitivo). Si no recuerdas los conceptos de las clases anteriores, el simulador te costará más. Estas 3 preguntas funcionan como diagnóstico rápido: te dicen si necesitas repasar antes de seguir.

Antes de entrar al simulador, veamos qué recuerdas del módulo.
Responde las 3 preguntas y presiona el botón para que la IA evalúe.

### 🐍 Python en esta celda

**`widgets.Textarea()`** — Crea un campo de texto donde puedes escribir. El parámetro `placeholder` es el texto gris de sugerencia. `layout` controla el tamaño visual.

**`widgets.Output()`** — Un área de salida donde el código puede mostrar resultados dinámicamente. Funciona como una "pantalla dentro de la pantalla".

**`def funcion(btn):`** — Función que se activa cuando presionas un botón. El parámetro `btn` es el botón mismo — por eso podemos hacer `btn.disabled = True` para desactivarlo después de hacer clic.

**`with out_recap1:`** — Bloque de contexto: todo lo que se imprima dentro de este bloque aparece en el widget `Output`, no en la salida normal.

**`clear_output()`** — Limpia el contenido previo del widget de salida antes de mostrar algo nuevo.

**`.strip()`** — Elimina espacios y saltos de línea al inicio y final de un texto. Útil para verificar si alguien escribió algo o dejó el campo vacío.

In [ ]:
#@title Pregunta 1: Nombra los 6 pasos de la metodología de Business Analytics
resp_recap1 = widgets.Textarea(
    placeholder='Escribe los 6 pasos...',
    layout=widgets.Layout(width='100%', height='80px')
)
out_recap1 = widgets.Output()

def eval_recap1(btn):
    btn.disabled = True
    with out_recap1:
        clear_output()
        if not resp_recap1.value.strip():
            print('⚠️ Escribe tu respuesta primero')
            btn.disabled = False
            return
        fb = ask_llm(
            'Eres evaluador de un curso de Business Analytics. Evalúa si el participante nombró correctamente los 6 pasos de la metodología BA. Empieza con ✅/⚠️/❌. Feedback máx 3 oraciones. Español.',
            f'Respuesta: {resp_recap1.value}', max_tokens=300
        )
        print(fb)
        pts = 2 if fb.startswith('✅') else 1 if fb.startswith('⚠️') else 0
        add_score('recap', pts, 2)
        show_score()

btn1 = widgets.Button(description='Evaluar', button_style='primary')
btn1.on_click(eval_recap1)
display(resp_recap1, btn1, out_recap1)

Textarea(value='', layout=Layout(height='80px', width='100%'), placeholder='Escribe los 6 pasos...')

Button(button_style='primary', description='Evaluar', style=ButtonStyle())

Output()

### 🐍 Python en esta celda

> Mismos conceptos que la Pregunta 1: `Textarea`, `Output`, función con botón, `ask_llm()`. La estructura se repite — ese es el patrón del simulador. Una vez que entiendes una pregunta, entiendes todas.

In [ ]:
#@title Pregunta 2: Da un ejemplo de sesgo de supervivencia en educación
resp_recap2 = widgets.Textarea(
    placeholder='Tu ejemplo aquí...',
    layout=widgets.Layout(width='100%', height='80px')
)
out_recap2 = widgets.Output()

def eval_recap2(btn):
    btn.disabled = True
    with out_recap2:
        clear_output()
        if not resp_recap2.value.strip():
            print('⚠️ Escribe tu respuesta primero')
            btn.disabled = False
            return
        fb = ask_llm(
            'Eres evaluador de un curso de BA. Evalúa si el ejemplo de sesgo de supervivencia en educación es correcto y relevante. Empieza con ✅/⚠️/❌. Máx 3 oraciones. Español.',
            f'Respuesta: {resp_recap2.value}', max_tokens=300
        )
        print(fb)
        pts = 2 if fb.startswith('✅') else 1 if fb.startswith('⚠️') else 0
        add_score('recap', pts, 2)
        show_score()

btn2 = widgets.Button(description='Evaluar', button_style='primary')
btn2.on_click(eval_recap2)
display(resp_recap2, btn2, out_recap2)

Textarea(value='', layout=Layout(height='80px', width='100%'), placeholder='Tu ejemplo aquí...')

Button(button_style='primary', description='Evaluar', style=ButtonStyle())

Output()

### 🐍 Python en esta celda

> Mismo patrón: campo de texto → botón → enviar a IA → mostrar feedback. Si entendiste la Pregunta 1, esta celda no tiene código nuevo.

In [ ]:
#@title Pregunta 3: Diferencia en una oración análisis descriptivo de predictivo
resp_recap3 = widgets.Textarea(
    placeholder='Una oración que diferencie ambos...',
    layout=widgets.Layout(width='100%', height='60px')
)
out_recap3 = widgets.Output()

def eval_recap3(btn):
    btn.disabled = True
    with out_recap3:
        clear_output()
        if not resp_recap3.value.strip():
            print('⚠️ Escribe tu respuesta primero')
            btn.disabled = False
            return
        fb = ask_llm(
            'Eres evaluador de un curso de BA. Evalúa si la diferenciación entre análisis descriptivo y predictivo es correcta y concisa. Empieza con ✅/⚠️/❌. Máx 3 oraciones. Español.',
            f'Respuesta: {resp_recap3.value}', max_tokens=300
        )
        print(fb)
        pts = 2 if fb.startswith('✅') else 1 if fb.startswith('⚠️') else 0
        add_score('recap', pts, 2)
        show_score()

btn3 = widgets.Button(description='Evaluar', button_style='primary')
btn3.on_click(eval_recap3)
display(resp_recap3, btn3, out_recap3)

Textarea(value='', layout=Layout(height='60px', width='100%'), placeholder='Una oración que diferencie ambos..…

Button(button_style='primary', description='Evaluar', style=ButtonStyle())

Output()

---
## 🔍 Módulo 1: Caso Deserción Estudiantil en FACES

> **¿Por qué un caso de deserción?** Porque es un problema real de FACES que puedes conectar con tu experiencia. Trabajar con un caso cercano te obliga a pensar como analista, no como estudiante que memoriza. La revelación progresiva (paso a paso, sin ver todo de golpe) simula cómo trabaja un analista de datos real: primero la pregunta, luego las variables, después el hallazgo, y al final la acción.

Vas a investigar este caso paso a paso, como un analista de datos real.
**No avances hasta completar cada paso.** La revelación progresiva es parte del aprendizaje.

> **¿Por qué empezar por la pregunta?** Todo análisis de datos empieza con una pregunta, nunca con los datos. Si empiezas por los datos, terminas haciendo «pesca de patrones» sin dirección. La pregunta define qué variables necesitas, qué método usarás y qué respuesta es útil.

### 🐍 Python en esta celda

**`display(HTML("""..."""))`** — Muestra contenido HTML formateado (con colores, tamaños, estilos) directamente en el notebook. Las triples comillas `"""` permiten escribir texto de varias líneas.

**HTML básico usado aquí:**
- `<div style='...'>` — Caja con estilo (color de fondo, bordes redondeados)
- `<h3>` — Título mediano
- `<p>` — Párrafo de texto
- `<b>` — Texto en negrita

In [ ]:
#@title Paso 1: Lee la pregunta de investigación
display(HTML("""
<div style='background:#1a237e;color:white;padding:20px;border-radius:10px;'>
    <h3 style='color:white;margin:0;'>❓ La Pregunta de Investigación</h3>
    <p style='font-size:18px;margin:10px 0 0 0;'>
        ¿Qué factores predicen la deserción en primer año de Economía en FACES?
    </p>
</div>
"""))
print('\n👇 En la siguiente celda, propón las variables que recopilarías para responder esta pregunta.')


👇 En la siguiente celda, propón las variables que recopilarías para responder esta pregunta.


> **¿Por qué proponerlas antes de verlas?** Para que actives tu propio criterio antes de recibir la respuesta. Si te doy las variables primero, tu cerebro solo las memoriza. Si tú las propones primero, luego puedes comparar y entender *qué se te escapó y por qué*. Eso es aprendizaje profundo.

### 🐍 Python en esta celda

**`widgets.Button(description='texto')`** — Crea un botón clickeable. `description` es lo que dice el botón.

**`.on_click(funcion)`** — Conecta un botón con una función: "cuando hagan clic en este botón, ejecuta esta función". Es el patrón fundamental de interactividad en notebooks.

**`btn.description = 'Evaluando...'`** — Puedes cambiar el texto de un botón *mientras el código se ejecuta*, dando feedback visual al usuario.

In [ ]:
#@title Paso 1b: Propón tus variables (mínimo 5)
txt_variables = widgets.Textarea(
    placeholder='Ej: promedio de notas, edad, si trabaja, ...\nUna variable por línea',
    layout=widgets.Layout(width='100%', height='120px')
)
out_p1 = widgets.Output()

def evaluar_p1(btn):
    btn.disabled = True
    btn.description = 'Evaluando...'
    with out_p1:
        clear_output()
        fb = ask_llm(
            """Eres evaluador de un ejercicio de identificación de variables para un estudio de deserción estudiantil.
Las variables reales del estudio son: Promedio 1er trimestre, Carrera, Turno (diurno/nocturno),
Edad de ingreso, ¿Trabaja?, Nota de ingreso (PAA), Asistencia (% clases), ¿Desertó? (variable dependiente).

Evalúa las variables propuestas:
1. ¿Cuántas coinciden con las reales?
2. ¿Propuso alguna variable interesante adicional?
3. ¿Olvidó alguna obvia?
Puntaje de 0-5. Español, conciso.""",
            f'Variables propuestas:\n{txt_variables.value}', max_tokens=500
        )
        print(fb)
        try:
            nums = re.findall(r'(\d+)\s*/\s*5', fb)
            pts = int(nums[0]) if nums else 3
        except: pts = 3
        add_score('caso_desercion', pts, 5)
    show_score()
    btn.description = '✓ Evaluado'

btn_p1 = widgets.Button(description='Enviar mis variables', button_style='primary', layout=widgets.Layout(width='250px'))
btn_p1.on_click(evaluar_p1)
display(txt_variables, btn_p1, out_p1)

Textarea(value='', layout=Layout(height='120px', width='100%'), placeholder='Ej: promedio de notas, edad, si t…

Button(button_style='primary', description='Enviar mis variables', layout=Layout(width='250px'), style=ButtonS…

Output()

> **¿Por qué revelar ahora?** Este es el momento de contraste: comparas lo que propusiste con lo que realmente se usó. Las diferencias son donde más aprendes. Además, clasificar variables por tipo y nivel de medición es una habilidad fundamental — determina qué análisis estadístico puedes hacer con cada una.

### 🐍 Python en esta celda

**Lista de tuplas** — `variables_reales = [('Promedio', 'Cuantitativa', 'Razón'), ...]` es una lista donde cada elemento es una *tupla* (grupo ordenado de valores). Ideal para datos con estructura fija (nombre, tipo, nivel).

**Ciclo `for` con desempaquetado** — `for nombre, tipo, nivel in variables_reales:` extrae automáticamente los tres valores de cada tupla en cada iteración.

**Construcción de HTML con ciclos** — Se construye una tabla HTML concatenando filas dentro de un `for`. Esto es más flexible que `print()` porque controlas colores, bordes y formato visual.

In [ ]:
#@title Paso 2: Revelación — Las variables reales del estudio
variables_reales = [
    ('Promedio 1er trimestre', 'Cuantitativa', 'Razón'),
    ('Carrera', 'Cualitativa', 'Nominal'),
    ('Turno (diurno/nocturno)', 'Cualitativa', 'Nominal'),
    ('Edad de ingreso', 'Cuantitativa', 'Razón'),
    ('¿Trabaja?', 'Cualitativa', 'Nominal'),
    ('Nota de ingreso (PAA)', 'Cuantitativa', 'Intervalo'),
    ('Asistencia (% clases)', 'Cuantitativa', 'Razón'),
    ('¿Desertó? (sí/no)', 'Cualitativa', 'Nominal'),
]

display(HTML('<h4>📋 Variables reales del estudio:</h4>'))
html_table = '<table style="border-collapse:collapse;width:100%;">'
html_table += '<tr style="background:#1a237e;color:white;"><th style="padding:8px;">Variable</th><th style="padding:8px;">Tipo</th><th style="padding:8px;">Nivel</th></tr>'
for nombre, tipo, nivel in variables_reales:
    html_table += f'<tr><td style="padding:6px;border-bottom:1px solid #ddd;">{nombre}</td><td style="padding:6px;border-bottom:1px solid #ddd;">{tipo}</td><td style="padding:6px;border-bottom:1px solid #ddd;">{nivel}</td></tr>'
html_table += '</table>'
display(HTML(html_table))
print(f'\n📊 Dataset: rendimiento_academico.csv | n = 2,847 registros | Fuente: datos sintéticos FACES')
print('\n👇 En la siguiente celda, clasifica tú mismo cada variable.')

Variable,Tipo,Nivel
Promedio 1er trimestre,Cuantitativa,Razón
Carrera,Cualitativa,Nominal
Turno (diurno/nocturno),Cualitativa,Nominal
Edad de ingreso,Cuantitativa,Razón
¿Trabaja?,Cualitativa,Nominal
Nota de ingreso (PAA),Cuantitativa,Intervalo
Asistencia (% clases),Cuantitativa,Razón
¿Desertó? (sí/no),Cualitativa,Nominal



📊 Dataset: rendimiento_academico.csv | n = 2,847 registros | Fuente: datos sintéticos FACES

👇 En la siguiente celda, clasifica tú mismo cada variable.


> **¿Por qué clasificar con dropdowns?** Porque leer la clasificación no es lo mismo que hacerla. Los dropdowns te obligan a decidir activamente si una variable es cuantitativa o cualitativa, nominal o de razón. Esto refuerza la taxonomía que vas a necesitar para tu propio proyecto en el Lab 01.

### 🐍 Python en esta celda

**Lista de diccionarios** — `[{'nombre': 'Promedio', 'tipo': 'Cuantitativa', 'nivel': 'Razón'}, ...]`. Cada diccionario es un registro con campos nombrados. Es la forma más natural de representar datos tabulares en Python.

**`widgets.Dropdown(options=[...], value='')`** — Crea un menú desplegable. `options` son las opciones disponibles, `value` es la selección inicial.

**`enumerate(lista)`** — Recorre una lista dando tanto el índice (posición) como el valor. `for i, v in enumerate(['a','b','c'])` da: `0,'a'` → `1,'b'` → `2,'c'`.

In [ ]:
#@title Paso 2b: Clasifica las variables (dropdowns)
variables_quiz = [
    {'nombre': 'Promedio 1er trimestre', 'tipo': 'Cuantitativa', 'nivel': 'Razón'},
    {'nombre': 'Carrera', 'tipo': 'Cualitativa', 'nivel': 'Nominal'},
    {'nombre': 'Turno (diurno/nocturno)', 'tipo': 'Cualitativa', 'nivel': 'Nominal'},
    {'nombre': 'Edad de ingreso', 'tipo': 'Cuantitativa', 'nivel': 'Razón'},
    {'nombre': '¿Trabaja?', 'tipo': 'Cualitativa', 'nivel': 'Nominal'},
    {'nombre': 'Nota de ingreso (PAA)', 'tipo': 'Cuantitativa', 'nivel': 'Intervalo'},
    {'nombre': 'Asistencia (% clases)', 'tipo': 'Cuantitativa', 'nivel': 'Razón'},
    {'nombre': '¿Desertó? (sí/no)', 'tipo': 'Cualitativa', 'nivel': 'Nominal'},
]

tipo_opts = ['Selecciona...', 'Cuantitativa', 'Cualitativa']
nivel_opts = ['Selecciona...', 'Nominal', 'Ordinal', 'Intervalo', 'Razón']
dd_tipo, dd_nivel = [], []
rows = [widgets.HBox([widgets.HTML('<b style="width:200px;display:inline-block">Variable</b>'),
                       widgets.HTML('<b style="width:150px;display:inline-block">Tipo</b>'),
                       widgets.HTML('<b style="width:150px;display:inline-block">Nivel</b>')])]

for v in variables_quiz:
    dt = widgets.Dropdown(options=tipo_opts, value='Selecciona...', layout=widgets.Layout(width='150px'))
    dn = widgets.Dropdown(options=nivel_opts, value='Selecciona...', layout=widgets.Layout(width='150px'))
    dd_tipo.append(dt); dd_nivel.append(dn)
    rows.append(widgets.HBox([widgets.HTML(f'<span style="width:200px;display:inline-block">{v["nombre"]}</span>'), dt, dn]))

out_p2 = widgets.Output()

def evaluar_p2(btn):
    btn.disabled = True
    correctas = 0
    total = len(variables_quiz) * 2
    with out_p2:
        clear_output()
        for i, v in enumerate(variables_quiz):
            tok = dd_tipo[i].value == v['tipo']
            nok = dd_nivel[i].value == v['nivel']
            correctas += int(tok) + int(nok)
            ti = '✅' if tok else '❌'
            ni = '✅' if nok else '❌'
            line = f"  {v['nombre']:30s} Tipo: {ti}"
            if not tok: line += f" (era {v['tipo']})"
            line += f"   Nivel: {ni}"
            if not nok: line += f" (era {v['nivel']})"
            print(line)
        pct = correctas / total * 100
        print(f'\n🎯 {correctas}/{total} correctas ({pct:.0f}%)')
        if pct < 70:
            print('\n💡 Tip: Razón tiene cero absoluto (edad, porcentaje). Intervalo no (puntajes estandarizados).')
        add_score('caso_desercion', round(correctas / total * 5), 5)
    show_score()
    btn.description = '✓ Evaluado'

btn_p2 = widgets.Button(description='Verificar clasificación', button_style='primary', layout=widgets.Layout(width='250px'))
btn_p2.on_click(evaluar_p2)
rows.extend([btn_p2, out_p2])
display(widgets.VBox(rows))

> **¿Por qué pedirte una predicción?** Para exponer tu intuición antes de ver los datos. La mayoría de las personas elige «si trabaja» o «nota de ingreso» — y se equivoca. Ese error es valioso: te enseña que la intuición sin datos es poco confiable, que es exactamente el argumento central de Business Analytics.

### 🐍 Python en esta celda

**`widgets.RadioButtons(options=[...])`** — Crea botones de opción donde solo puedes elegir uno. Perfecto para preguntas de selección única.

**`.value`** — Propiedad que contiene la opción actualmente seleccionada. `prediccion.value` te dice qué eligió el usuario.

In [ ]:
#@title Paso 3: ¿Qué variable predice mejor la deserción? — Haz tu predicción
display(HTML("""
<div style='background:#303f9f;color:white;padding:15px;border-radius:10px;margin-bottom:15px;'>
    <h3 style='color:white;margin:0;'>🎯 Haz tu predicción</h3>
    <p style='margin:5px 0 0 0;'>De las 7 variables independientes, ¿cuál es el <b>mejor predictor</b> de deserción?</p>
</div>
"""))

prediccion = widgets.RadioButtons(
    options=['Promedio 1er trimestre', 'Carrera', 'Turno (diurno/nocturno)',
             'Edad de ingreso', '¿Trabaja?', 'Nota de ingreso (PAA)', 'Asistencia (% clases)'],
    value=None, layout=widgets.Layout(width='100%')
)
txt_justif = widgets.Textarea(placeholder='¿Por qué crees que esa variable es la más predictiva?',
                               layout=widgets.Layout(width='100%', height='60px'))
display(widgets.HTML('<b>Selecciona:</b>'), prediccion)
display(widgets.HTML('<b>Justifica:</b>'), txt_justif)
print('\n👇 Ejecuta la siguiente celda para revelar el hallazgo.')

HTML(value='<b>Selecciona:</b>')

RadioButtons(layout=Layout(width='100%'), options=('Promedio 1er trimestre', 'Carrera', 'Turno (diurno/nocturn…

HTML(value='<b>Justifica:</b>')

Textarea(value='', layout=Layout(height='60px', width='100%'), placeholder='¿Por qué crees que esa variable es…


👇 Ejecuta la siguiente celda para revelar el hallazgo.


> **¿Por qué revelar después de predecir?** El contraste entre tu predicción y el hallazgo real genera sorpresa cognitiva — un estado mental donde el cerebro presta máxima atención. Si acertaste, se refuerza tu razonamiento. Si fallaste, el error se ancla mejor que cualquier explicación teórica.

### 🐍 Python en esta celda

**Operador `==`** — Comparación: `eleccion == 'Promedio 1er trimestre'` devuelve `True` o `False`.

**Operador ternario** — `'✅' if correcta else '⚠️'` es un *if* en una sola línea: "si `correcta` es True, usa '✅'; si no, usa '⚠️'". Se usa mucho dentro de f-strings para texto condicional.

**f-string con expresiones** — Puedes poner expresiones completas dentro de `{}`: `f"{'Correcto' if x else 'Incorrecto'}"` evalúa la condición *dentro* del string.

In [ ]:
#@title Paso 3b: Revelación del hallazgo
eleccion = prediccion.value
correcta = eleccion == 'Promedio 1er trimestre'

display(HTML(f"""
<div style='background:{"#4CAF50" if correcta else "#FF9800"};color:white;padding:20px;border-radius:10px;margin:10px 0;'>
    <h3 style='color:white;'>{'✅ ¡Correcto!' if correcta else '⚠️ No exactamente...'}</h3>
    <p>Tu predicción: <b>{eleccion}</b></p>
</div>
<div style='background:#1b5e20;color:white;padding:20px;border-radius:10px;margin:10px 0;'>
    <h2 style='color:white;text-align:center;'>3× más riesgo de deserción</h2>
    <p style='text-align:center;font-size:16px;'>si el promedio del 1er trimestre es menor a 10/20</p>
</div>
<h4>Hallazgos clave:</h4>
<ul>
    <li><b>Promedio del 1er trimestre</b> = mejor predictor (no la nota de ingreso ni el empleo)</li>
    <li><b>Asistencia < 60%</b> multiplica el riesgo por 2.1×</li>
    <li><b>Trabajar NO fue significativo</b> (p=0.34) — sorprendente</li>
    <li>El turno nocturno tiene efecto, pero <b>desaparece al controlar por asistencia</b></li>
</ul>
<p><b>💡 Lección:</b> Cuando controlas por asistencia, el efecto de trabajar desaparece.
La variable real es asistencia, no empleo. El turno nocturno es <i>mediador</i>, no causa.</p>
"""))

add_score('caso_desercion', 5 if correcta else 2, 5)
show_score()

> **¿Por qué feedback de IA y no solo la respuesta correcta?** Porque importa *cómo razonaste*, no solo si acertaste. La IA analiza tu justificación y te dice si tu lógica fue correcta aunque la respuesta no, o si acertaste por las razones equivocadas. Eso es metacognición aplicada.

### 🐍 Python en esta celda

**`if valor.strip():`** — Verifica que haya texto antes de enviarlo a la IA. `.strip()` elimina espacios; si el resultado está vacío, Python lo trata como `False`.

**Prompt engineering** — El string largo que empieza con `'Eres profesor de metodología...'` es el *system prompt*: instrucciones que le dicen a la IA cómo comportarse y qué evaluar. Es la misma técnica que usan ChatGPT, Claude y todos los asistentes de IA.

In [ ]:
#@title Paso 3c: Feedback personalizado de la IA sobre tu justificación
if txt_justif.value.strip():
    fb = ask_llm(
        """Eres profesor de metodología. El participante predijo qué variable predice mejor la deserción.
Respuesta correcta: Promedio del 1er trimestre (3x riesgo si < 10/20).
Trabajar NO es significativo (p=0.34). Asistencia < 60% = 2.1x riesgo.
Si eligió 'trabajar' o 'nota de ingreso', explica por qué esas intuiciones son comunes pero incorrectas.
Máx 4 oraciones. Español.""",
        f'Eligió: {prediccion.value}\nJustificación: {txt_justif.value}', max_tokens=400
    )
    print(f'🤖 Feedback:\n{fb}')
else:
    print('⚠️ No escribiste justificación — la IA no puede dar feedback personalizado.')

⚠️ No escribiste justificación — la IA no puede dar feedback personalizado.


> **¿Por qué pasar de hallazgo a acción?** Porque un dato sin decisión es solo trivia. El ciclo completo de BA es: Pregunta → Datos → Análisis → Hallazgo → **Acción**. Este paso te entrena para cerrar el ciclo — la habilidad más escasa y más valiosa en las organizaciones.

### 🐍 Python en esta celda

> Mismo patrón: `Textarea` → `Button` → `ask_llm()` → `Output`. La novedad aquí es conceptual (pasar de hallazgo a acción), no técnica. El código reutiliza exactamente las mismas piezas que ya conoces.

In [ ]:
#@title Paso 4: De hallazgo a acción — ¿Qué harías como decano?
display(HTML("""
<div style='background:#4527a0;color:white;padding:15px;border-radius:10px;margin-bottom:15px;'>
    <h3 style='color:white;margin:0;'>🎬 Pregunta → Datos → Análisis → Hallazgo → ¿ACCIÓN?</h3>
    <p style='margin:5px 0 0 0;'>Eres el decano de FACES. Propón <b>3 acciones concretas</b> basadas en los datos.</p>
</div>
"""))

txt_acciones = widgets.Textarea(
    placeholder='Acción 1: ...\nAcción 2: ...\nAcción 3: ...',
    layout=widgets.Layout(width='100%', height='120px')
)
out_p4 = widgets.Output()

def evaluar_p4(btn):
    btn.disabled = True
    btn.description = 'Evaluando...'
    with out_p4:
        clear_output()
        fb = ask_llm(
            """Eres evaluador experto en intervenciones educativas basadas en datos.
Hallazgos: Promedio 1er trimestre < 10/20 = 3x riesgo. Asistencia < 60% = 2.1x. Trabajar no significativo.
Acciones reales implementadas:
1. Alerta temprana en semana 4 (promedio < 10 y asistencia < 60% activa protocolo)
2. Tutorías focalizadas a 15-20 estudiantes de mayor riesgo
3. Rediseño de evaluación continua en 1er trimestre

Compara con las propuestas del participante. ¿Son coherentes con los datos? Puntaje 0-5. Español.""",
            f'Acciones propuestas:\n{txt_acciones.value}', max_tokens=500
        )
        print(fb)
        print('\n' + '='*60)
        print('📋 ACCIONES REALES IMPLEMENTADAS:')
        print('='*60)
        print('1. Alerta temprana en semana 4: promedio < 10 y asistencia < 60% → protocolo de acompañamiento')
        print('2. Tutorías focalizadas: tutor asignado a los 15-20 de mayor riesgo')
        print('3. Evaluación continua en 1er trimestre para detectar dificultades antes del parcial')
        try:
            nums = re.findall(r'(\d+)\s*/\s*5', fb)
            pts = int(nums[0]) if nums else 3
        except: pts = 3
        add_score('caso_desercion', pts, 5)
    show_score()
    btn.description = '✓ Caso completo'

btn_p4 = widgets.Button(description='Evaluar mis acciones', button_style='primary', layout=widgets.Layout(width='250px'))
btn_p4.on_click(evaluar_p4)
display(txt_acciones, btn_p4, out_p4)

Textarea(value='', layout=Layout(height='120px', width='100%'), placeholder='Acción 1: ...\nAcción 2: ...\nAcc…

Button(button_style='primary', description='Evaluar mis acciones', layout=Layout(width='250px'), style=ButtonS…

Output()

---
## 🕵️ Módulo 2: Detective de Sesgos — Inflación en América Latina

> **¿Por qué un módulo de sesgos con datos de inflación?** Porque la manipulación visual de datos es el arma más peligrosa del mundo corporativo y político. El mismo dataset puede sustentar conclusiones opuestas según cómo lo grafiques. Si no aprendes a detectar estas manipulaciones, vas a tomar decisiones basándote en gráficos diseñados para engañarte.

El mismo dataset puede contar historias completamente opuestas.
Tu trabajo: identificar las manipulaciones.

> **¿Por qué?** Necesitamos un dataset compartido para construir tres versiones del mismo gráfico. Usar datos de inflación de LATAM los hace reconocibles — ya los has visto en noticias. Eso hace que las manipulaciones que verás sean más impactantes y fáciles de recordar.

### 🐍 Python en esta celda

**`np.random.seed(42)`** — Fija la semilla del generador de números aleatorios. Con la misma semilla, los datos "aleatorios" salen siempre iguales. `42` es una convención (referencia a *The Hitchhiker's Guide to the Galaxy*).

**`list(range(2000, 2025))`** — Genera una lista de números del 2000 al 2024 (el último no se incluye). `range()` es la forma estándar de crear secuencias numéricas.

**`pd.DataFrame(diccionario, index=lista)`** — Crea una tabla (DataFrame) a partir de un diccionario donde cada clave es una columna y cada valor es una lista de datos. El `index` define las etiquetas de las filas (aquí, los años).

In [ ]:
#@title Preparar datos de inflación
np.random.seed(42)
years = list(range(2000, 2025))

inflation_data = {
    'Chile': [3.8,3.6,2.5,2.8,1.1,3.1,3.4,4.4,8.7,1.5,1.4,3.3,3.0,1.8,4.7,4.3,3.8,2.2,2.4,2.3,3.0,4.5,11.6,7.6,4.2],
    'Colombia': [9.2,8.0,6.3,7.1,5.9,5.0,4.3,5.5,7.0,4.2,2.3,3.4,3.2,2.0,2.9,5.0,7.5,4.3,3.2,3.5,2.5,5.6,13.1,9.3,5.6],
    'México': [9.5,6.4,5.0,4.5,4.7,4.0,3.6,3.8,5.1,5.3,4.2,3.4,4.1,3.8,4.0,2.7,2.8,6.0,4.9,3.6,3.4,7.4,7.9,4.7,4.2],
    'Argentina': [25.0,25.0,41.0,3.7,6.1,12.3,9.8,8.5,7.2,7.7,10.9,9.5,10.8,10.9,38.0,26.5,41.0,25.7,47.6,53.8,42.0,50.9,94.8,211.0,120.0],
    'Venezuela': [16.2,12.5,22.4,31.1,21.7,16.0,13.7,18.7,30.4,28.6,29.1,26.1,21.1,40.6,62.2,121.7,254.9,438.1,65374.0,19906.0,2959.8,686.4,234.1,337.5,100.0]
}

df_inf = pd.DataFrame(inflation_data, index=years)
print(f'✅ Datos de inflación cargados: {len(df_inf)} años, {len(df_inf.columns)} países')
df_inf.tail()

✅ Datos de inflación cargados: 25 años, 5 países


,Chile,Colombia,México,Argentina,Venezuela
2020,3.0,2.5,3.4,42.0,2959.8
2021,4.5,5.6,7.4,50.9,686.4
2022,11.6,13.1,7.9,94.8,234.1
2023,7.6,9.3,4.7,211.0,337.5
2024,4.2,5.6,4.2,120.0,100.0


> **¿Por qué esta versión?** Para mostrarte cómo una selección sesgada de países (solo Argentina y Venezuela), un rango temporal corto y una escala manipulada del eje Y pueden fabricar una narrativa alarmista. Todos estos son trucos comunes en medios y reportes corporativos.

### 🐍 Python en esta celda

**`go.Figure()`** — Crea un gráfico vacío de Plotly. Luego le agregas *trazas* (líneas, barras, puntos) con `.add_trace()`.

**`go.Scatter(x=..., y=..., mode='lines+markers')`** — Agrega una línea con puntos. `x` son los valores del eje horizontal, `y` del vertical. `mode` controla si muestra líneas, puntos o ambos.

**`.update_layout()`** — Modifica la apariencia general del gráfico: título, rango de ejes, plantilla visual, altura.

**`df_inf.loc[2022:2024, 'Argentina']`** — `.loc[]` selecciona filas y columnas por etiqueta (no por posición). Aquí selecciona los años 2022 a 2024 de la columna Argentina.

**`.values`** — Convierte una columna de pandas a un array simple de numpy, que Plotly necesita para graficar.

In [ ]:
#@title Versión A: «La inflación se disparó»
fig_a = go.Figure()
for country in ['Argentina', 'Venezuela']:
    fig_a.add_trace(go.Scatter(x=list(range(2022, 2025)), y=df_inf.loc[2022:2024, country].values,
                               mode='lines+markers', name=country, line=dict(width=3)))
fig_a.update_layout(title="Versión A: «La inflación se disparó»",
                    yaxis_title='Inflación (%)', xaxis_title='Año',
                    yaxis_range=[50, 350], template='plotly_white', height=350)
fig_a.show()
print('🤔 ¿Qué manipulaciones ves en esta gráfica?')

🤔 ¿Qué manipulaciones ves en esta gráfica?


In [ ]:
import plotly.express as px

# Selecciona un país para graficar su inflación
country_to_plot = 'Venezuela'

# Crea un DataFrame temporal para el país seleccionado
df_country_inflation = df_inf[[country_to_plot]].reset_index()
df_country_inflation.columns = ['Año', 'Inflación (%)']

# Crea el gráfico de líneas
fig = px.line(df_country_inflation, x='Año', y='Inflación (%)',
              title=f'Inflación Anual en {country_to_plot}',
              labels={'Año': 'Año', 'Inflación (%)': 'Tasa de Inflación (%)'})

fig.update_layout(template='plotly_white', hovermode='x unified')
fig.show()


> **¿Por qué una versión opuesta?** Para que veas el mismo dataset contando la historia contraria. Aquí se excluyen los países con alta inflación y se usa una escala Y que aplana las variaciones. El contraste entre A y B es la lección central: los datos no mienten, pero las visualizaciones sí pueden.

### 🐍 Python en esta celda

> Mismo patrón que la Versión A (`Figure` → `add_trace` → `update_layout`), pero con diferentes países y un rango de eje Y (`yaxis_range=[0, 100]`) que aplana las variaciones. La diferencia es editorial, no técnica.

In [ ]:
#@title Versión B: «La inflación está controlada»
fig_b = go.Figure()
for country in ['Chile', 'Colombia', 'México']:
    fig_b.add_trace(go.Scatter(x=years, y=df_inf[country].values,
                               mode='lines+markers', name=country, line=dict(width=2)))
fig_b.update_layout(title="Versión B: «La inflación está controlada»",
                    yaxis_title='Inflación (%)', xaxis_title='Año',
                    yaxis_range=[0, 100], template='plotly_white', height=350)
fig_b.show()
print('🤔 ¿Qué manipulaciones ves en esta gráfica?')

> **¿Por qué una versión «honesta»?** Para darte un modelo de referencia. Separar los datos en paneles por escala, mostrar todos los países y usar el rango temporal completo es lo que un analista ético haría. Esto te da criterios concretos para evaluar cualquier gráfico.

### 🐍 Python en esta celda

**`make_subplots(rows=1, cols=2)`** — Crea una figura con múltiples paneles (subgráficos). Aquí: 1 fila, 2 columnas = dos gráficos lado a lado.

**`row=1, col=1` / `row=1, col=2`** — Al agregar trazas, especificas en qué panel va cada una. Panel izquierdo = inflación moderada, panel derecho = hiperinflación.

**`subplot_titles=('Título 1', 'Título 2')`** — Títulos individuales para cada panel.

In [ ]:
#@title Versión C: «Depende del contexto» — La versión honesta
fig_c = make_subplots(rows=1, cols=2, subplot_titles=('Inflación moderada', 'Hiperinflación'))
for country, color in [('Chile','#2196F3'),('Colombia','#4CAF50'),('México','#FF9800')]:
    fig_c.add_trace(go.Scatter(x=years, y=df_inf[country].values, mode='lines', name=country,
                               line=dict(color=color, width=2)), row=1, col=1)
for country, color in [('Argentina','#f44336'),('Venezuela','#9C27B0')]:
    fig_c.add_trace(go.Scatter(x=years, y=df_inf[country].values, mode='lines', name=country,
                               line=dict(color=color, width=2)), row=1, col=2)
fig_c.update_yaxes(title_text='%', range=[0, 20], row=1, col=1)
fig_c.update_yaxes(title_text='%', type='log', row=1, col=2)
fig_c.add_hline(y=3, line_dash='dash', line_color='gray', annotation_text='Meta ~3%', row=1, col=1)
fig_c.update_layout(title="Versión C: «Depende del contexto»", template='plotly_white', height=400)
fig_c.show()
print('✅ Paneles separados, escala apropiada por grupo, meta de inflación marcada.')
print('\n👇 Ahora identifica las manipulaciones de cada versión.')

✅ Paneles separados, escala apropiada por grupo, meta de inflación marcada.

👇 Ahora identifica las manipulaciones de cada versión.


> **¿Por qué checkboxes?** Para que practiques la detección activa. No basta con saber que existen sesgos — tienes que poder señalarlos en un gráfico específico. Los checkboxes te obligan a comprometerte con una respuesta antes de ver la solución.

### 🐍 Python en esta celda

**`widgets.Checkbox(value=False, description='texto')`** — Crea una casilla de verificación. `value=False` significa que empieza desmarcada. El usuario puede marcar o desmarcar cada una.

**Lista de widgets** — `checks_a = [Checkbox(...), Checkbox(...), ...]` agrupa varios widgets en una lista. Después puedes recorrerlos con `for` para leer cuáles fueron marcados.

In [ ]:
#@title Identifica las manipulaciones — Versión A
display(HTML('<h4>Versión A: «La inflación se disparó» — ¿Qué manipulaciones tiene?</h4>'))

checks_a = [
    widgets.Checkbox(value=False, description='Manipulación de escala del eje Y'),
    widgets.Checkbox(value=False, description='Cherry-picking de países'),
    widgets.Checkbox(value=False, description='Selección sesgada del período temporal'),
    widgets.Checkbox(value=False, description='Exclusión de datos incómodos'),
    widgets.Checkbox(value=False, description='Escala que aplana la variación'),
]
# Correctas: 0 (escala), 1 (cherry-pick), 2 (período)
for cb in checks_a:
    display(cb)
print('\n👇 Ahora la Versión B en la siguiente celda.')

> **¿Por qué repetir el ejercicio?** Porque cada versión usa manipulaciones *diferentes*. La versión A exagera; la B minimiza. Detectar ambas direcciones de sesgo es lo que distingue a un pensador crítico de alguien que solo detecta lo que confirma sus sospechas.

### 🐍 Python en esta celda

> Mismo patrón de checkboxes que la Versión A. Cada versión tiene su propia lista `checks_b` con las mismas opciones pero respuestas correctas diferentes.

In [ ]:
#@title Identifica las manipulaciones — Versión B
display(HTML('<h4>Versión B: «La inflación está controlada» — ¿Qué manipulaciones tiene?</h4>'))

checks_b = [
    widgets.Checkbox(value=False, description='Manipulación de escala del eje Y'),
    widgets.Checkbox(value=False, description='Cherry-picking de países'),
    widgets.Checkbox(value=False, description='Selección sesgada del período temporal'),
    widgets.Checkbox(value=False, description='Exclusión de datos incómodos'),
    widgets.Checkbox(value=False, description='Escala que aplana la variación'),
]
# Correctas: 3 (exclusión), 4 (escala aplana)
for cb in checks_b:
    display(cb)

> **¿Por qué preguntar si C es honesta?** Para introducir un matiz crucial: «honesto» no significa «perfecto». Incluso la versión C tiene decisiones editoriales. La transparencia total no existe — pero hay grados, y distinguirlos es la competencia que estás desarrollando.

### 🐍 Python en esta celda

**`widgets.RadioButtons(options=[...], value=None)`** — Botones de opción. `value=None` significa que ninguno está seleccionado al inicio — el usuario debe elegir activamente.

In [ ]:
#@title Versión C — ¿Es honesta?
check_c = widgets.RadioButtons(
    options=['También tiene manipulaciones', 'Es la versión honesta/transparente'],
    value=None
)
display(HTML('<h4>Versión C — ¿Qué opinas?</h4>'))
display(check_c)
print('\n👇 Ejecuta la siguiente celda para verificar tus respuestas.')

RadioButtons(options=('También tiene manipulaciones', 'Es la versión honesta/transparente'), value=None)


👇 Ejecuta la siguiente celda para verificar tus respuestas.


> **¿Por qué verificación automática?** Para darte feedback inmediato. La investigación en aprendizaje muestra que el feedback es más efectivo cuanto más cercano al momento de la decisión. Esperar días para saber si detectaste bien un sesgo reduce el impacto.

### 🐍 Python en esta celda

**Conjuntos (`set`)** — `{0, 1, 2}` es un conjunto: colección sin duplicados ni orden. Útil para comparar respuestas.

**Operadores de conjuntos:**
- `a_correct & a_selected` — **Intersección**: elementos que están en *ambos* conjuntos (respuestas correctas que el usuario marcó).
- `a_selected - a_correct` — **Diferencia**: elementos en `a_selected` que NO están en `a_correct` (falsos positivos).

**`len()`** — Cuenta cuántos elementos tiene una colección.

**`max(0, valor)`** — Asegura que el resultado no sea negativo. Si el usuario marcó más incorrectas que correctas, el puntaje queda en 0, no en negativo.

In [ ]:
#@title Verificar respuestas de sesgos
pts = 0

# Versión A: correctas = escala (0), cherry-pick (1), período (2)
a_correct = {0, 1, 2}
a_selected = {i for i, cb in enumerate(checks_a) if cb.value}
match_a = len(a_correct & a_selected)
false_a = len(a_selected - a_correct)
pts_a = max(0, match_a - false_a)
print(f'Versión A: {pts_a}/{len(a_correct)} correctas')
print(f'  ✓ Manipulación de escala + Cherry-picking de países + Selección sesgada del período')

# Versión B: correctas = exclusión (3), escala aplana (4)
b_correct = {3, 4}
b_selected = {i for i, cb in enumerate(checks_b) if cb.value}
match_b = len(b_correct & b_selected)
false_b = len(b_selected - b_correct)
pts_b = max(0, match_b - false_b)
print(f'\nVersión B: {pts_b}/{len(b_correct)} correctas')
print(f'  ✓ Exclusión de datos incómodos + Escala que aplana la variación')

# Versión C
c_ok = check_c.value == 'Es la versión honesta/transparente'
print(f'\nVersión C: {"✅ Correcto" if c_ok else "❌ Incorrecto"} — Es la versión transparente')

total_pts = pts_a + pts_b + (1 if c_ok else 0)
total_max = len(a_correct) + len(b_correct) + 1
add_score('sesgos', round(total_pts / total_max * 5), 5)

print(f'\n💡 Lección: El MISMO dataset puede contar historias opuestas.')
print('   La honestidad está en la transparencia metodológica, no en los datos.')
show_score()

> **¿Por qué construir tu propio gráfico?** Porque la mejor forma de entender la manipulación visual es intentar hacer una visualización honesta tú mismo. Al elegir países, rango de años y tipo de escala, experimentas en primera persona las decisiones que un analista toma.

### 🐍 Python en esta celda

**`widgets.SelectMultiple(options=[...], value=[...])`** — Lista donde puedes seleccionar *varios* elementos (a diferencia de `Dropdown` que solo permite uno). `value` define la selección inicial.

**`widgets.IntRangeSlider(value=[2000, 2024], min=..., max=...)`** — Control deslizante con dos manijas para seleccionar un rango. El usuario arrastra para elegir año de inicio y fin.

**`widgets.Dropdown(options=[...])`** — Menú desplegable de selección única.

**`widgets.VBox([w1, w2, w3])`** — Apila widgets verticalmente. `HBox` los pone horizontalmente. Son contenedores de layout.

In [ ]:
#@title BONUS: Construye tu propia visualización honesta
display(HTML('<h4>🛠️ Elige los parámetros para una visualización honesta</h4>'))

paises_sel = widgets.SelectMultiple(
    options=['Chile', 'Colombia', 'México', 'Argentina', 'Venezuela'],
    value=['Chile', 'Colombia', 'México'], description='Países:',
    layout=widgets.Layout(width='300px', height='120px'))
rango_anios = widgets.IntRangeSlider(value=[2000, 2024], min=2000, max=2024, step=1,
                                      description='Período:', layout=widgets.Layout(width='500px'))
escala_y = widgets.Dropdown(options=['Automática', '0-20%', '0-100%', 'Logarítmica'],
                            value='Automática', description='Escala Y:')
mostrar_meta = widgets.Checkbox(value=True, description='Mostrar meta de inflación (3%)')
out_custom = widgets.Output()

def generar_grafico(btn):
    with out_custom:
        clear_output()
        paises = list(paises_sel.value)
        yr_min, yr_max = rango_anios.value
        if not paises: print('⚠️ Selecciona al menos un país'); return
        fig = go.Figure()
        yrs = list(range(yr_min, yr_max+1))
        for c in paises:
            fig.add_trace(go.Scatter(x=yrs, y=df_inf.loc[yr_min:yr_max, c].values, mode='lines+markers', name=c))
        if mostrar_meta.value:
            fig.add_hline(y=3, line_dash='dash', line_color='gray', annotation_text='Meta ~3%')
        ya = {}
        if escala_y.value == '0-20%': ya = dict(range=[0, 20])
        elif escala_y.value == '0-100%': ya = dict(range=[0, 100])
        elif escala_y.value == 'Logarítmica': ya = dict(type='log')
        fig.update_layout(title='Tu visualización', yaxis_title='Inflación (%)', template='plotly_white', height=400, **(dict(yaxis=ya) if ya else {}))
        fig.show()
        fb = ask_llm(
            'Eres experto en visualización de datos. Evalúa las decisiones del participante para un gráfico de inflación. ¿Selección representativa? ¿Período adecuado? ¿Escala honesta? 3-4 oraciones. Español.',
            f'Países: {paises}, Período: {yr_min}-{yr_max}, Escala: {escala_y.value}, Meta visible: {mostrar_meta.value}', max_tokens=400
        )
        print(f'\n🤖 Evaluación:\n{fb}')

btn_custom = widgets.Button(description='Generar y evaluar', button_style='success', layout=widgets.Layout(width='250px'))
btn_custom.on_click(generar_grafico)
display(paises_sel, rango_anios, escala_y, mostrar_meta, btn_custom, out_custom)

---
## 🚀 Módulo 2.5: Datos Reales de Venezuela — Colab en Acción

> **¿Por qué datos en vivo de APIs?** Para que veas que los datos no viven en CSVs estáticos — viven en APIs que puedes consultar con código. Este módulo te muestra el superpoder de Colab: jalar, limpiar y visualizar datos reales en segundos, sin descargar archivos manualmente. Es la diferencia entre Excel y pensamiento computacional.

Aquí es donde Colab brilla. Vamos a **jalar datos reales** de Venezuela desde APIs públicas,
limpiarlos, visualizarlos y pedirle a la IA que los interprete. Sin descargar nada manualmente.

**Fuentes:** Banco Mundial (API), Our World in Data (GitHub), ACNUR/R4V (migración)

> **¿Por qué el Banco Mundial?** Porque es una fuente pública, confiable y con API abierta. Aprender a consultar APIs es una competencia transferible: el mismo patrón funciona para datos del BCV, CEPAL, FMI o cualquier organismo. Además, descargar datos con código hace tu análisis reproducible.

### 🐍 Python en esta celda

**`requests.get(url)`** — Hace una solicitud HTTP GET a una URL (como cuando tu navegador pide una página web). Aquí le pide datos al API del Banco Mundial.

**`.json()`** — Convierte la respuesta del servidor a un diccionario de Python. JSON (*JavaScript Object Notation*) es el formato estándar para intercambiar datos en la web.

**Diccionarios como mapeo** — `indicators` mapea códigos técnicos (`'NY.GDP.PCAP.CD'`) a nombres legibles (`'PIB per cápita'`). `countries` mapea códigos ISO a nombres.

**`f'https://api.worldbank.org/.../{variable}/...'`** — URL construida dinámicamente con f-string. El API del Banco Mundial espera el código del indicador y los países en la URL.

**`pd.DataFrame(lista_de_diccionarios)`** — Cada diccionario de la lista se convierte en una fila del DataFrame. Es la forma estándar de convertir datos de APIs a tablas.

In [ ]:
#@title Descargar datos del Banco Mundial — 4 indicadores, 5 países
indicators = {
    'NY.GDP.PCAP.CD': 'PIB per cápita (USD)',
    'SP.DYN.LE00.IN': 'Esperanza de vida (años)',
    'SE.PRM.ENRR': 'Matrícula escolar primaria (%)',
    'SL.UEM.TOTL.ZS': 'Desempleo (% fuerza laboral)'
}
countries = {'VEN': 'Venezuela', 'COL': 'Colombia', 'CHL': 'Chile', 'ECU': 'Ecuador', 'PER': 'Perú'}
country_codes = ';'.join(countries.keys())

print('⏳ Descargando datos del Banco Mundial...')
all_data = {}

for code_ind, name in indicators.items():
    url = f'https://api.worldbank.org/v2/country/{country_codes}/indicator/{code_ind}?format=json&per_page=500&date=2000:2023'
    try:
        r = requests.get(url, timeout=15)
        data = r.json()
        if len(data) > 1 and data[1]:
            rows = [{'pais': countries.get(d['country']['id'], d['country']['value']),
                     'year': int(d['date']), 'valor': d['value']}
                    for d in data[1] if d['value'] is not None]
            all_data[name] = pd.DataFrame(rows)
            print(f'  ✅ {name}: {len(rows)} registros')
    except Exception as e:
        print(f'  ❌ {name}: {e}')

print(f'\n📊 {sum(len(df) for df in all_data.values())} registros totales descargados en vivo')

> **¿Por qué PIB per cápita como primer indicador?** Porque es el indicador más intuitivo para comparar trayectorias económicas entre países. La divergencia de Venezuela respecto a la región es visualmente impactante y prepara el terreno para analizar indicadores sociales derivados.

### 🐍 Python en esta celda

**Filtrado de DataFrame** — `df_pib[df_pib['pais'] == 'Venezuela']` selecciona solo las filas donde la columna `pais` tiene el valor `'Venezuela'`. Es la operación más importante de pandas.

**`.sort_values('year')`** — Ordena las filas por la columna `year`. Crucial para que las líneas del gráfico no se crucen aleatoriamente.

**Diccionario de colores** — `colors = {'Venezuela': '#f44336', ...}` asigna un color hexadecimal a cada país. Mantener colores consistentes entre gráficos es buena práctica visual.

**`.get(pais, 'gray')`** — Busca el color del país; si no lo encuentra, usa gris como valor por defecto. Más seguro que `colors[pais]` que daría error si falta una clave.

In [ ]:
#@title Visualizar: PIB per cápita — Venezuela vs. la región
if 'PIB per cápita (USD)' in all_data:
    df_pib = all_data['PIB per cápita (USD)']
    fig = go.Figure()
    colors = {'Venezuela': '#f44336', 'Colombia': '#4CAF50', 'Chile': '#2196F3', 'Ecuador': '#FF9800', 'Perú': '#9C27B0'}
    for pais in df_pib['pais'].unique():
        dp = df_pib[df_pib['pais'] == pais].sort_values('year')
        fig.add_trace(go.Scatter(x=dp['year'], y=dp['valor'], mode='lines+markers', name=pais,
                                 line=dict(color=colors.get(pais, 'gray'), width=3 if pais == 'Venezuela' else 1.5)))
    fig.add_vrect(x0=2013, x1=2023, fillcolor='red', opacity=0.05, annotation_text='Crisis', annotation_position='top left')
    fig.update_layout(title='PIB per cápita: Venezuela vs. la región (Banco Mundial)', yaxis_title='USD',
                      template='plotly_white', height=450, hovermode='x unified')
    fig.show()
    print('💡 Venezuela pasó de liderar la región a tener el PIB per cápita más bajo.')
else:
    print('⚠️ Ejecuta la celda anterior primero.')

💡 Venezuela pasó de liderar la región a tener el PIB per cápita más bajo.


> **¿Por qué un dashboard de 4 indicadores?** Para practicar análisis multidimensional. Un solo indicador puede engañar (¿recuerdas el Módulo 2?). Ver PIB, esperanza de vida, educación y desempleo juntos te da una imagen más completa y te entrena para buscar relaciones entre variables.

### 🐍 Python en esta celda

**`make_subplots(rows=2, cols=2)`** — Grilla de 2×2 = 4 paneles. Cada indicador va en un panel.

**`enumerate(list(all_data.items())[:4])`** — Doble truco: `.items()` convierte el diccionario en pares (nombre, dataframe); `[:4]` toma solo los primeros 4; `enumerate` agrega un índice de posición.

**`positions = [(1,1), (1,2), (2,1), (2,2)]`** — Lista de coordenadas de panel. `positions[idx]` da la posición del panel correspondiente.

**Ciclos anidados** — Un `for` dentro de otro: el externo recorre indicadores, el interno recorre países. Cada combinación (indicador × país) agrega una línea al panel correspondiente.

In [ ]:
#@title Dashboard: 4 indicadores — Venezuela vs. la región
if len(all_data) >= 2:
    fig = make_subplots(rows=2, cols=2, subplot_titles=list(all_data.keys())[:4], vertical_spacing=0.12, horizontal_spacing=0.1)
    positions = [(1,1), (1,2), (2,1), (2,2)]
    colors = {'Venezuela': '#f44336', 'Colombia': '#4CAF50', 'Chile': '#2196F3', 'Ecuador': '#FF9800', 'Perú': '#9C27B0'}

    for idx, (name, df) in enumerate(list(all_data.items())[:4]):
        r, c = positions[idx]
        for pais in df['pais'].unique():
            dp = df[df['pais'] == pais].sort_values('year')
            fig.add_trace(go.Scatter(x=dp['year'], y=dp['valor'], mode='lines', name=pais,
                                     line=dict(color=colors.get(pais, 'gray'), width=3 if pais == 'Venezuela' else 1),
                                     legendgroup=pais, showlegend=(idx == 0)), row=r, col=c)
        fig.add_vline(x=2013, line_dash='dash', line_color='rgba(0,0,0,0.3)', row=r, col=c)

    fig.update_layout(height=600, template='plotly_white', title='Venezuela vs. la región: 4 dimensiones',
                      legend=dict(orientation='h', yanchor='bottom', y=-0.15, x=0.5, xanchor='center'))
    fig.show()
    print('💡 La línea punteada marca 2013. Venezuela diverge en todas las dimensiones.')
else:
    print('⚠️ Ejecuta la descarga de datos primero.')

> **¿Por qué una tabla antes/después?** Porque las tablas complementan a los gráficos: dan valores exactos que las líneas no muestran. Además, el contraste antes/después de la crisis te entrena para definir períodos de referencia — una decisión metodológica clave en cualquier análisis temporal.

### 🐍 Python en esta celda

**`print(f'{texto:<35s}')`** — Formato de alineación: `<35s` alinea a la izquierda con 35 caracteres de ancho. `>10s` alinea a la derecha. Esto crea columnas tabulares con `print()`.

**Filtrado con condiciones múltiples** — `ven[(ven['year'] >= 2010) & (ven['year'] <= 2014)]` usa `&` (AND lógico) para combinar condiciones. Cada condición va entre paréntesis.

**`.mean()`** — Calcula el promedio de una columna. Aquí promedia los valores del período pre-crisis (2010-2014) y post-crisis (2020+) para comparar.

In [ ]:
#@title Tabla comparativa: Venezuela antes y después de la crisis
print('='*70)
print('📋 VENEZUELA: ANTES vs DESPUÉS DE LA CRISIS')
print('='*70)
print(f'{"Indicador":<35s} {"~2012":>10s} {"Último":>10s} {"Cambio":>10s}')
print('-'*70)
for name, df in all_data.items():
    ven = df[df['pais'] == 'Venezuela'].sort_values('year')
    pre = ven[(ven['year'] >= 2010) & (ven['year'] <= 2014)]
    post = ven[ven['year'] >= 2020]
    if len(pre) > 0 and len(post) > 0:
        v_pre = pre['valor'].mean()
        v_post = post['valor'].mean()
        cambio = ((v_post - v_pre) / v_pre * 100)
        signo = '+' if cambio > 0 else ''
        print(f'{name:<35s} {v_pre:>10.1f} {v_post:>10.1f} {signo}{cambio:>8.1f}%')
print('\n💡 Estos son datos REALES descargados hace segundos del Banco Mundial.')

> **¿Por qué pedirle a la IA que interprete?** Para mostrarte un flujo de trabajo real: tú descargas y preparas los datos, la IA te ayuda a articular hallazgos y generar hipótesis. También para que practiques pensamiento crítico: ¿la IA dijo algo que los datos no respaldan?

### 🐍 Python en esta celda

**Concatenación de strings en ciclo** — `stats_summary += f'...'` acumula texto en cada iteración. Al final, `stats_summary` contiene un resumen de todos los indicadores, listo para enviárselo a la IA.

**`.iloc[0]` / `.iloc[-1]`** — Acceso por posición (no por etiqueta). `.iloc[0]` = primera fila, `.iloc[-1]` = última fila. Aquí extrae el primer y último valor de cada serie temporal.

**`:.1f`** — Formato numérico dentro de f-string: 1 decimal fijo. `f'{3.14159:.1f}'` → `'3.1'`.

In [ ]:
#@title La IA interpreta los datos que acabas de descargar
# Preparar resumen estadístico
stats_summary = ''
for name, df in all_data.items():
    ven = df[df['pais'] == 'Venezuela'].sort_values('year')
    if len(ven) > 0:
        stats_summary += f'\n{name}:\n'
        stats_summary += f'  Venezuela: {ven.iloc[0]["year"]}={ven.iloc[0]["valor"]:.1f} → {ven.iloc[-1]["year"]}={ven.iloc[-1]["valor"]:.1f}\n'
        for pais in ['Colombia', 'Chile']:
            p = df[df['pais'] == pais].sort_values('year')
            if len(p) > 0:
                stats_summary += f'  {pais}: {p.iloc[0]["year"]}={p.iloc[0]["valor"]:.1f} → {p.iloc[-1]["year"]}={p.iloc[-1]["valor"]:.1f}\n'

display(HTML('<h4>🤖 Pregúntale a la IA sobre los datos que descargaste</h4>'))

pregunta_ia = widgets.Textarea(
    value='¿Cuál fue el punto de quiebre del PIB per cápita de Venezuela y qué lo causó?',
    layout=widgets.Layout(width='100%', height='60px'))
out_ia = widgets.Output()

def consultar_ia(btn):
    btn.disabled = True; btn.description = 'Analizando...'
    with out_ia:
        clear_output()
        resp = ask_llm(
            f'Eres analista económico experto en América Latina. Datos REALES del Banco Mundial:\n{stats_summary}\n'
            'Responde basándote SOLO en los datos. Si no alcanzan, di qué falta. '
            'Formato: análisis (máx 6 oraciones), luego "Datos que lo sustentan:" con números. Español.',
            pregunta_ia.value, max_tokens=600)
        display(Markdown(resp))
    btn.disabled = False; btn.description = 'Preguntar a la IA'

btn_ia = widgets.Button(description='Preguntar a la IA', button_style='success', layout=widgets.Layout(width='200px'))
btn_ia.on_click(consultar_ia)
display(pregunta_ia, btn_ia, out_ia)

> **¿Por qué datos de migración?** Porque conectan los indicadores macroeconómicos con impacto humano. Los números de PIB y desempleo son abstractos; 7.7 millones de migrantes es concreto. Además, trabajar con datos de ACNUR te expone a una fuente distinta al Banco Mundial.

### 🐍 Python en esta celda

**Diccionario con listas paralelas** — `{'País': [...], 'Venezolanos': [...], 'lat': [...], 'lon': [...]}`. Cada lista tiene la misma longitud; la posición `i` en cada lista corresponde al mismo registro. `pd.DataFrame()` las convierte en columnas.

**`.sum()`** — Suma todos los valores de una columna. `df_mig['Venezolanos'].sum()` da el total de migrantes.

**`f'{numero:,.0f}'`** — Formato con separador de miles y sin decimales. `f'{2875743:,.0f}'` → `'2,875,743'`.

In [ ]:
#@title Migración venezolana — Datos ACNUR/R4V
migration_data = {
    'País': ['Colombia','Perú','Brasil','Ecuador','Chile','EE.UU.','España','Argentina','Rep. Dominicana','Panamá','México','Trinidad y Tobago','Otros'],
    'Venezolanos': [2875743,1542004,568270,475268,444423,545200,430000,171050,115283,148600,95000,35800,453359],
    'lat': [4.57,-12.04,-14.24,-1.83,-33.45,38.91,40.42,-34.60,18.49,8.98,19.43,10.65,0],
    'lon': [-74.30,-77.03,-51.93,-78.18,-70.67,-77.04,-3.70,-58.38,-69.93,-79.52,-99.13,-61.25,0]
}
df_mig = pd.DataFrame(migration_data)
total_mig = df_mig['Venezolanos'].sum()
print(f'📊 Total: {total_mig:,.0f} venezolanos fuera del país (R4V/ACNUR 2025)')
print(f'\nTop 5 países receptores:')
for _, r in df_mig.nlargest(5, 'Venezolanos').iterrows():
    pct = r['Venezolanos'] / total_mig * 100
    print(f"  {r['País']:20s} {r['Venezolanos']:>12,.0f}  ({pct:.1f}%)")

> **¿Por qué un mapa?** Porque la distribución geográfica se entiende mejor en un mapa que en una tabla. Este tipo de visualización (burbujas proporcionales) es estándar en análisis de flujos migratorios y te enseña que elegir el tipo de gráfico correcto es parte del análisis.

### 🐍 Python en esta celda

**`go.Scattergeo(lat=..., lon=...)`** — Coloca puntos en un mapa geográfico. `lat` y `lon` son coordenadas. A diferencia de `go.Scatter` (gráfico X-Y), `Scattergeo` proyecta sobre un mapa real.

**`marker=dict(size=..., color=...)`** — Controla el tamaño y color de cada punto. Aquí el tamaño es proporcional al número de migrantes (`Venezolanos/30000`) y el color usa una escala (`colorscale='Reds'`).

**`lambda r: f"{r['País']}: {r['Venezolanos']:,.0f}"`** — Función anónima (*lambda*) que genera el texto de cada punto. `.apply(lambda)` la aplica a cada fila del DataFrame.

In [ ]:
#@title Mapa: ¿Hacia dónde migran los venezolanos?
df_mig_plot = df_mig[df_mig['País'] != 'Otros']

fig_map = go.Figure()
fig_map.add_trace(go.Scattergeo(
    lat=df_mig_plot['lat'], lon=df_mig_plot['lon'],
    text=df_mig_plot.apply(lambda r: f"{r['País']}: {r['Venezolanos']:,.0f}", axis=1),
    marker=dict(size=df_mig_plot['Venezolanos']/30000, color=df_mig_plot['Venezolanos'],
                colorscale='Reds', showscale=True, colorbar=dict(title='Personas'), sizemin=5),
    mode='markers+text', textposition='top center', textfont=dict(size=9)))

fig_map.add_trace(go.Scattergeo(lat=[7.0], lon=[-66.0], text=['Venezuela'],
    marker=dict(size=15, color='gold', symbol='star'), mode='markers+text',
    textposition='bottom center', showlegend=False))

fig_map.update_geos(scope='world', showcountries=True, countrycolor='lightgray',
                    lataxis_range=[-55, 50], lonaxis_range=[-120, 10], bgcolor='#f0f8ff')
fig_map.update_layout(title='Diáspora venezolana (ACNUR/R4V 2025)', height=500, margin=dict(t=50, b=20, l=20, r=20))
fig_map.show()

> **¿Por qué un treemap además del mapa?** Porque cada tipo de gráfico resalta algo diferente. El mapa muestra *dónde*; el treemap muestra *proporciones relativas*. Ver que Colombia sola recibe ~40% de la diáspora es más evidente en un treemap que en burbujas sobre mapa.

### 🐍 Python en esta celda

**`px.treemap(df, path=['País'], values='Venezolanos')`** — Crea un treemap (mapa de árbol) con una sola línea. `path` define las categorías, `values` define el tamaño de cada rectángulo. Plotly Express (`px`) simplifica lo que en `go` requeriría muchas líneas.

**Cálculo de porcentaje** — `df_mig[df_mig['País'] == 'Colombia']['Venezolanos'].iloc[0] / df_mig['Venezolanos'].sum() * 100`: filtra → toma el valor → divide por el total → multiplica por 100.

In [ ]:
#@title Treemap: Distribución proporcional
fig_tree = px.treemap(df_mig, path=['País'], values='Venezolanos',
                      color='Venezolanos', color_continuous_scale='RdYlBu_r',
                      title='Distribución de venezolanos por país receptor')
fig_tree.update_layout(height=400)
fig_tree.show()
col_pct = df_mig[df_mig['País'] == 'Colombia']['Venezolanos'].iloc[0] / df_mig['Venezolanos'].sum() * 100
print(f'💡 Colombia sola recibe el {col_pct:.1f}% de toda la diáspora.')
print('\n🤔 Pregunta para reflexionar: ¿Qué datos necesitarías para predecir el flujo migratorio de los próximos 2 años?')

💡 Colombia sola recibe el 36.4% de toda la diáspora.

🤔 Pregunta para reflexionar: ¿Qué datos necesitarías para predecir el flujo migratorio de los próximos 2 años?


> **¿Por qué preguntas abiertas?** Para que pases de consumir análisis a generarlos. Hasta ahora seguiste instrucciones; ahora tú decides qué preguntar. Esta transición de guiado a autónomo es deliberada — replica cómo funciona un analista real después del entrenamiento.

### 🐍 Python en esta celda

**`widgets.Dropdown(options=[...])`** — Menú con preguntas sugeridas. La primera opción permite escribir tu propia pregunta.

**Comprensión de lista** — `'\n'.join([f"  {r['País']}: {r['Venezolanos']:,.0f}" for _, r in df_mig.iterrows()])` es un ciclo comprimido en una línea que genera un string formateado por cada fila del DataFrame. `.join()` los une con saltos de línea.

In [ ]:
#@title Desafío: Hazle preguntas a TODOS los datos con IA
display(HTML('<h4>🧠 Tienes datos del Banco Mundial + migración ACNUR. Pregunta lo que quieras.</h4>'))

migration_summary = '\n'.join([f"  {r['País']}: {r['Venezolanos']:,.0f}" for _, r in df_mig.iterrows()])

sugerencias = widgets.Dropdown(
    options=['(Escribe tu propia pregunta)',
             '¿Hay correlación entre la caída del PIB y el flujo migratorio?',
             '¿Qué país de la región tuvo la mejor trayectoria en la última década?',
             '¿La esperanza de vida bajó en Venezuela? ¿Cuánto vs la región?',
             'Si fueras asesor del gobierno, ¿qué 3 indicadores priorizarías?'],
    value='(Escribe tu propia pregunta)', layout=widgets.Layout(width='100%'))

pregunta_libre = widgets.Textarea(placeholder='Tu pregunta analítica...', layout=widgets.Layout(width='100%', height='60px'))
def sync(change):
    if change['new'] != '(Escribe tu propia pregunta)': pregunta_libre.value = change['new']
sugerencias.observe(sync, names='value')

out_desafio = widgets.Output()

def analizar(btn):
    btn.disabled = True; btn.description = 'Analizando...'
    with out_desafio:
        clear_output()
        if not pregunta_libre.value.strip(): print('⚠️ Escribe una pregunta'); btn.disabled = False; btn.description = 'Analizar con IA'; return
        resp = ask_llm(
            f'Eres analista de datos experto en Venezuela y América Latina.\n'
            f'Datos REALES disponibles:\n\nBANCO MUNDIAL (2000-2023):\n{stats_summary}\n'
            f'MIGRACIÓN (ACNUR/R4V 2025):\n{migration_summary}\nTotal: ~7.9M\n\n'
            'Responde: 1) Análisis directo (3-4 oraciones) 2) Datos que sustentan 3) Limitaciones 4) Para profundizar. Español.',
            pregunta_libre.value, max_tokens=800)
        display(Markdown(resp))
        add_score('datos_reales', 3, 3)
    show_score()
    btn.disabled = False; btn.description = 'Analizar con IA'

btn_desafio = widgets.Button(description='Analizar con IA', button_style='warning', layout=widgets.Layout(width='200px'))
btn_desafio.on_click(analizar)
display(sugerencias, pregunta_libre, btn_desafio, out_desafio)

---
## 📝 Módulo 3: Lab 01 — Traduce tu Asignatura a Datos

> **¿Por qué traducir *tu* asignatura?** Porque el objetivo del curso no es que analices datos de otros — es que aprendas a convertir *tus propias* preguntas profesionales en proyectos de datos. Este lab es donde todo lo anterior se vuelve personal: aplicas la metodología BA a tu contexto real.

Ahora es personal. Vas a diseñar un proyecto de datos para **tu propia asignatura**.
Completa cada sección y la IA te da feedback antes de avanzar.

> **¿Por qué empezar por el contexto?** Porque sin contexto, cualquier análisis es genérico. Saber cuántos estudiantes tienes, qué decisiones tomas sin datos y qué te frustra es lo que convierte una pregunta vaga en una pregunta analítica precisa.

### 🐍 Python en esta celda

> Patrón ya conocido: `display(HTML(...))` para el encabezado visual + `widgets.Textarea()` para el campo de texto + `widgets.Button()` → `ask_llm()` para feedback. A partir de aquí, el código no introduce conceptos nuevos — aplica los mismos bloques que ya dominas.

In [ ]:
#@title Lab 01 — Sección 1: Mi asignatura y contexto
display(HTML("""
<div style='background:#1565c0;color:white;padding:10px 15px;border-radius:8px;'>
    <h4 style='color:white;margin:0;'>1. Mi asignatura y contexto</h4>
    <p style='margin:3px 0 0 0;font-size:13px;'>Carrera, semestre, # estudiantes, ¿qué decisiones tomas sin datos hoy?</p>
</div>"""))

lab_1 = widgets.Textarea(placeholder='Ej: Imparto Estadística I en Economía, 4to semestre, ~45 estudiantes.',
    layout=widgets.Layout(width='100%', height='100px'))
out_lab_1 = widgets.Output()

def eval_lab_1(btn):
    btn.disabled = True; btn.description = 'Evaluando...'
    with out_lab_1:
        clear_output()
        fb = ask_llm(
            'Eres mentor de Business Analytics. Evalúa esta sección del Lab 01 de un docente universitario.\n'
            'Criterio: Evalúa si el contexto es claro y específico. ¿Se entiende la asignatura y las decisiones sin datos? Sugiere mejoras.\n'
            'Formato: ✅/⚠️/❌ + feedback máx 4 oraciones + Puntaje: X/5. Español.',
            f'Sección: Mi asignatura y contexto\nRespuesta: {lab_1.value}', max_tokens=500)
        print(fb)
        try:
            nums = re.findall(r'(\d+)\s*/\s*5', fb)
            pts = int(nums[0]) if nums else 3
        except: pts = 3
        add_score('lab01', pts, 5)
    show_score()
    btn.description = '✓ Evaluado'

btn_lab_1 = widgets.Button(description='Evaluar sección 1', button_style='info', layout=widgets.Layout(width='200px'))
btn_lab_1.on_click(eval_lab_1)
display(lab_1, btn_lab_1, out_lab_1)

> **¿Por qué una pregunta analítica?** Porque es el paso 1 de la metodología BA y el más difícil. Una buena pregunta es específica, medible y accionable. Si tu pregunta es vaga («¿cómo mejorar mi clase?»), tu análisis será vago.

### 🐍 Python en esta celda

> Mismo patrón que Sección 1. Cada sección del Lab usa la misma estructura: encabezado HTML → campo de texto → botón de evaluación → feedback de IA.

In [ ]:
#@title Lab 01 — Sección 2: Pregunta analítica
display(HTML("""
<div style='background:#1565c0;color:white;padding:10px 15px;border-radius:8px;'>
    <h4 style='color:white;margin:0;'>2. Pregunta analítica</h4>
    <p style='margin:3px 0 0 0;font-size:13px;'>Específica, medible, relevante para mejorar tu asignatura.</p>
</div>"""))

lab_2 = widgets.Textarea(placeholder='Ej: ¿Qué factores del primer parcial predicen la reprobación final?',
    layout=widgets.Layout(width='100%', height='100px'))
out_lab_2 = widgets.Output()

def eval_lab_2(btn):
    btn.disabled = True; btn.description = 'Evaluando...'
    with out_lab_2:
        clear_output()
        fb = ask_llm(
            'Eres mentor de Business Analytics. Evalúa esta sección del Lab 01 de un docente universitario.\n'
            'Criterio: Evalúa si la pregunta es específica, medible y relevante. Si es demasiado amplia o estrecha, sugiere cómo ajustarla.\n'
            'Formato: ✅/⚠️/❌ + feedback máx 4 oraciones + Puntaje: X/5. Español.',
            f'Sección: Pregunta analítica\nRespuesta: {lab_2.value}', max_tokens=500)
        print(fb)
        try:
            nums = re.findall(r'(\d+)\s*/\s*5', fb)
            pts = int(nums[0]) if nums else 3
        except: pts = 3
        add_score('lab01', pts, 5)
    show_score()
    btn.description = '✓ Evaluado'

btn_lab_2 = widgets.Button(description='Evaluar sección 2', button_style='info', layout=widgets.Layout(width='200px'))
btn_lab_2.on_click(eval_lab_2)
display(lab_2, btn_lab_2, out_lab_2)

> **¿Por qué listar variables con clasificación?** Porque es exactamente lo que hiciste en el caso de deserción, pero ahora para tu proyecto. La clasificación (tipo + nivel de medición) determina qué análisis estadísticos puedes hacer después.

### 🐍 Python en esta celda

> Mismo patrón. La repetición es deliberada: cuando el código se vuelve predecible, puedes enfocarte en el *contenido* (tus variables), no en la *mecánica*.

In [ ]:
#@title Lab 01 — Sección 3: Variables necesarias
display(HTML("""
<div style='background:#1565c0;color:white;padding:10px 15px;border-radius:8px;'>
    <h4 style='color:white;margin:0;'>3. Variables necesarias</h4>
    <p style='margin:3px 0 0 0;font-size:13px;'>Lista con clasificación: nombre, tipo, nivel de medición.</p>
</div>"""))

lab_3 = widgets.Textarea(placeholder='Ej: Nota parcial 1 (Cuantitativa, Razón), Turno (Cualitativa, Nominal)',
    layout=widgets.Layout(width='100%', height='100px'))
out_lab_3 = widgets.Output()

def eval_lab_3(btn):
    btn.disabled = True; btn.description = 'Evaluando...'
    with out_lab_3:
        clear_output()
        fb = ask_llm(
            'Eres mentor de Business Analytics. Evalúa esta sección del Lab 01 de un docente universitario.\n'
            'Criterio: Evalúa: ¿Las variables responden la pregunta? ¿Clasificación correcta? ¿Falta variable dependiente? Corrige errores.\n'
            'Formato: ✅/⚠️/❌ + feedback máx 4 oraciones + Puntaje: X/5. Español.',
            f'Sección: Variables necesarias\nRespuesta: {lab_3.value}', max_tokens=500)
        print(fb)
        try:
            nums = re.findall(r'(\d+)\s*/\s*5', fb)
            pts = int(nums[0]) if nums else 3
        except: pts = 3
        add_score('lab01', pts, 5)
    show_score()
    btn.description = '✓ Evaluado'

btn_lab_3 = widgets.Button(description='Evaluar sección 3', button_style='info', layout=widgets.Layout(width='200px'))
btn_lab_3.on_click(eval_lab_3)
display(lab_3, btn_lab_3, out_lab_3)

> **¿Por qué definir la fuente?** Porque muchos proyectos de datos mueren aquí: diseñas un análisis brillante pero los datos no existen o no puedes acceder a ellos. Definir la fuente temprano te obliga a ser realista.

### 🐍 Python en esta celda

> Mismo patrón de las secciones anteriores.

In [ ]:
#@title Lab 01 — Sección 4: Fuente de datos
display(HTML("""
<div style='background:#1565c0;color:white;padding:10px 15px;border-radius:8px;'>
    <h4 style='color:white;margin:0;'>4. Fuente de datos</h4>
    <p style='margin:3px 0 0 0;font-size:13px;'>¿De dónde vendrán? Encuesta, base institucional, datos públicos.</p>
</div>"""))

lab_4 = widgets.Textarea(placeholder='Ej: Registros del sistema académico + encuesta breve a estudiantes.',
    layout=widgets.Layout(width='100%', height='100px'))
out_lab_4 = widgets.Output()

def eval_lab_4(btn):
    btn.disabled = True; btn.description = 'Evaluando...'
    with out_lab_4:
        clear_output()
        fb = ask_llm(
            'Eres mentor de Business Analytics. Evalúa esta sección del Lab 01 de un docente universitario.\n'
            'Criterio: Evalúa: ¿La fuente es accesible? ¿Tamaño de muestra suficiente? ¿Consideraciones éticas?\n'
            'Formato: ✅/⚠️/❌ + feedback máx 4 oraciones + Puntaje: X/5. Español.',
            f'Sección: Fuente de datos\nRespuesta: {lab_4.value}', max_tokens=500)
        print(fb)
        try:
            nums = re.findall(r'(\d+)\s*/\s*5', fb)
            pts = int(nums[0]) if nums else 3
        except: pts = 3
        add_score('lab01', pts, 5)
    show_score()
    btn.description = '✓ Evaluado'

btn_lab_4 = widgets.Button(description='Evaluar sección 4', button_style='info', layout=widgets.Layout(width='200px'))
btn_lab_4.on_click(eval_lab_4)
display(lab_4, btn_lab_4, out_lab_4)

> **¿Por qué elegir el tipo de análisis?** Porque descriptivo, exploratorio, inferencial y predictivo no son intercambiables — cada uno responde preguntas diferentes y requiere datos diferentes. Elegir mal aquí significa hacer un análisis que no responde tu pregunta.

### 🐍 Python en esta celda

> Mismo patrón de las secciones anteriores.

In [ ]:
#@title Lab 01 — Sección 5: Tipo de análisis
display(HTML("""
<div style='background:#1565c0;color:white;padding:10px 15px;border-radius:8px;'>
    <h4 style='color:white;margin:0;'>5. Tipo de análisis</h4>
    <p style='margin:3px 0 0 0;font-size:13px;'>¿Descriptivo? ¿Exploratorio? ¿Inferencial? ¿Predictivo? Justifica.</p>
</div>"""))

lab_5 = widgets.Textarea(placeholder='Ej: Predictivo — identificar tempranamente estudiantes en riesgo.',
    layout=widgets.Layout(width='100%', height='100px'))
out_lab_5 = widgets.Output()

def eval_lab_5(btn):
    btn.disabled = True; btn.description = 'Evaluando...'
    with out_lab_5:
        clear_output()
        fb = ask_llm(
            'Eres mentor de Business Analytics. Evalúa esta sección del Lab 01 de un docente universitario.\n'
            'Criterio: Evalúa coherencia entre pregunta y tipo de análisis. Corrige si hay confusión entre tipos.\n'
            'Formato: ✅/⚠️/❌ + feedback máx 4 oraciones + Puntaje: X/5. Español.',
            f'Sección: Tipo de análisis\nRespuesta: {lab_5.value}', max_tokens=500)
        print(fb)
        try:
            nums = re.findall(r'(\d+)\s*/\s*5', fb)
            pts = int(nums[0]) if nums else 3
        except: pts = 3
        add_score('lab01', pts, 5)
    show_score()
    btn.description = '✓ Evaluado'

btn_lab_5 = widgets.Button(description='Evaluar sección 5', button_style='info', layout=widgets.Layout(width='200px'))
btn_lab_5.on_click(eval_lab_5)
display(lab_5, btn_lab_5, out_lab_5)

> **¿Por qué anticipar sesgos?** Porque todo proyecto de datos tiene sesgos — la pregunta es si los ves o no. Listarlos *antes* de recopilar datos te permite diseñar mitigaciones. Este es el paso que separa un análisis ingenuo de uno riguroso.

### 🐍 Python en esta celda

> Mismo patrón de las secciones anteriores.

In [ ]:
#@title Lab 01 — Sección 6: Sesgos potenciales
display(HTML("""
<div style='background:#1565c0;color:white;padding:10px 15px;border-radius:8px;'>
    <h4 style='color:white;margin:0;'>6. Sesgos potenciales</h4>
    <p style='margin:3px 0 0 0;font-size:13px;'>Al menos 2 sesgos + mitigación.</p>
</div>"""))

lab_6 = widgets.Textarea(placeholder='Ej: Sesgo de supervivencia (solo datos de quienes terminaron). Mitigación: incluir registros de retiro.',
    layout=widgets.Layout(width='100%', height='100px'))
out_lab_6 = widgets.Output()

def eval_lab_6(btn):
    btn.disabled = True; btn.description = 'Evaluando...'
    with out_lab_6:
        clear_output()
        fb = ask_llm(
            'Eres mentor de Business Analytics. Evalúa esta sección del Lab 01 de un docente universitario.\n'
            'Criterio: Evalúa: ¿Sesgos reales y relevantes? ¿Mitigaciones prácticas? ¿Olvidó sesgo de su posición como docente?\n'
            'Formato: ✅/⚠️/❌ + feedback máx 4 oraciones + Puntaje: X/5. Español.',
            f'Sección: Sesgos potenciales\nRespuesta: {lab_6.value}', max_tokens=500)
        print(fb)
        try:
            nums = re.findall(r'(\d+)\s*/\s*5', fb)
            pts = int(nums[0]) if nums else 3
        except: pts = 3
        add_score('lab01', pts, 5)
    show_score()
    btn.description = '✓ Evaluado'

btn_lab_6 = widgets.Button(description='Evaluar sección 6', button_style='info', layout=widgets.Layout(width='200px'))
btn_lab_6.on_click(eval_lab_6)
display(lab_6, btn_lab_6, out_lab_6)

> **¿Por qué un resumen con feedback de IA?** Para cerrar el ciclo de diseño con una revisión externa. La IA evalúa la coherencia interna de tu proyecto: ¿la pregunta se puede responder con las variables que propusiste? ¿Las fuentes son viables?

### 🐍 Python en esta celda

**`eval(f'lab_{i}.value')`** — `eval()` ejecuta un string como si fuera código Python. `eval('lab_1.value')` es equivalente a escribir `lab_1.value`. Aquí se usa para recorrer `lab_1` a `lab_6` en un ciclo sin repetir código. *Nota: `eval()` es potente pero riesgoso en producción — aquí está controlado.*

**`try: ... except: pass`** — Intenta ejecutar algo; si falla, lo ignora silenciosamente (`pass` = "no hagas nada"). Aquí previene errores si alguna sección no fue completada.

In [ ]:
#@title Generar resumen del Lab 01
display(HTML('<h4>📋 Cuando hayas completado las 6 secciones, genera tu resumen:</h4>'))
out_resumen = widgets.Output()

def generar_resumen(btn):
    btn.disabled = True; btn.description = 'Generando...'
    with out_resumen:
        clear_output()
        proyecto = ''
        for i in range(1, 7):
            try:
                val = eval(f'lab_{i}.value')
                proyecto += f'Sección {i}: {val}\n\n'
            except: pass
        resumen = ask_llm(
            'Eres mentor de Business Analytics. Genera un resumen ejecutivo del proyecto:\n'
            '## Resumen\n**Asignatura:** ...\n**Pregunta:** ...\n**Variables:** (tabla markdown)\n'
            '**Fuente:** ...\n**Análisis:** ...\n**Sesgos:** ...\n\n'
            '## Evaluación global\n- Fortalezas (2-3)\n- Mejoras (2-3)\n- Recomendación para Módulo 2\n'
            '## Puntaje: X/10\nEspañol, constructivo.',
            f'Proyecto:\n{proyecto}', max_tokens=1000)
        display(Markdown(resumen))
    btn.description = '✓ Generado'

btn_resumen = widgets.Button(description='Generar resumen Lab 01', button_style='success', layout=widgets.Layout(width='300px'))
btn_resumen.on_click(generar_resumen)
display(btn_resumen, out_resumen)

---
## 🧪 Módulo 4: Quiz Integrador

> **¿Por qué un quiz al final?** Para consolidación — el cerebro retiene mejor cuando tiene que recuperar información poco después de aprenderla (*retrieval practice*). Las preguntas son deliberadamente ambiguas porque en la vida real los problemas no vienen con opciones claras. No es evaluativo: es diagnóstico.

**No es evaluativo — es diagnóstico.** Las preguntas son ambiguas a propósito.

### 🐍 Python en esta celda

> Las preguntas del quiz reutilizan el mismo patrón que ya conoces del Recap y el Lab: `Textarea` → `Button` → `ask_llm()` → `add_score()` → `show_score()`. No hay código nuevo — solo preguntas más desafiantes.

In [ ]:
#@title Quiz — Pregunta 1
display(HTML("<div style='background:#37474f;color:white;padding:10px;border-radius:6px;'>"
             f"<b>Pregunta 1:</b> En la metodología BA, ¿cuál paso es el más importante y por qué?</div>"))

quiz_1 = widgets.Textarea(placeholder='Tu respuesta...', layout=widgets.Layout(width='100%', height='70px'))
out_q1 = widgets.Output()

def eval_q1(btn):
    btn.disabled = True
    with out_q1:
        clear_output()
        if not quiz_1.value.strip(): print('⚠️ Sin respuesta'); add_score('quiz', 0, 2); show_score(); return
        fb = ask_llm(
            'Evalúa esta respuesta de BA. Correcta/esperada: Trampa: todos son necesarios, pero el paso 1 (objetivo y palancas) determina todo. Si defines mal el objetivo, el mejor análisis es inútil. '
            'Formato: ✅/⚠️/❌ + 2-3 oraciones. Si captura la idea central, es correcta. Español.',
            f'Pregunta: En la metodología BA, ¿cuál paso es el más importante y por qué?\nRespuesta: {quiz_1.value}', max_tokens=300)
        print(fb)
        pts = 2 if fb.startswith('✅') else 1 if fb.startswith('⚠️') else 0
        add_score('quiz', pts, 2)
        show_score()

btn_q1 = widgets.Button(description='Evaluar', button_style='primary')
btn_q1.on_click(eval_q1)
display(quiz_1, btn_q1, out_q1)

In [ ]:
#@title Quiz — Pregunta 2
display(HTML("<div style='background:#37474f;color:white;padding:10px;border-radius:6px;'>"
             f"<b>Pregunta 2:</b> Un estudio muestra que los países con más camas de hospital tienen mayor mortalidad. ¿Qué sesgo aplica?</div>"))

quiz_2 = widgets.Textarea(placeholder='Tu respuesta...', layout=widgets.Layout(width='100%', height='70px'))
out_q2 = widgets.Output()

def eval_q2(btn):
    btn.disabled = True
    with out_q2:
        clear_output()
        if not quiz_2.value.strip(): print('⚠️ Sin respuesta'); add_score('quiz', 0, 2); show_score(); return
        fb = ask_llm(
            'Evalúa esta respuesta de BA. Correcta/esperada: Causalidad espuria / variable confusora: países con más camas tienen población más envejecida. '
            'Formato: ✅/⚠️/❌ + 2-3 oraciones. Si captura la idea central, es correcta. Español.',
            f'Pregunta: Un estudio muestra que los países con más camas de hospital tienen mayor mortalidad. ¿Qué sesgo aplica?\nRespuesta: {quiz_2.value}', max_tokens=300)
        print(fb)
        pts = 2 if fb.startswith('✅') else 1 if fb.startswith('⚠️') else 0
        add_score('quiz', pts, 2)
        show_score()

btn_q2 = widgets.Button(description='Evaluar', button_style='primary')
btn_q2.on_click(eval_q2)
display(quiz_2, btn_q2, out_q2)

In [ ]:
#@title Quiz — Pregunta 3
display(HTML("<div style='background:#37474f;color:white;padding:10px;border-radius:6px;'>"
             f"<b>Pregunta 3:</b> «El 95% de nuestros graduados consigue empleo en 6 meses.» ¿Qué preguntarías?</div>"))

quiz_3 = widgets.Textarea(placeholder='Tu respuesta...', layout=widgets.Layout(width='100%', height='70px'))
out_q3 = widgets.Output()

def eval_q3(btn):
    btn.disabled = True
    with out_q3:
        clear_output()
        if not quiz_3.value.strip(): print('⚠️ Sin respuesta'); add_score('quiz', 0, 2); show_score(); return
        fb = ask_llm(
            'Evalúa esta respuesta de BA. Correcta/esperada: Sesgo de supervivencia + selección: ¿A cuántos contactaron? ¿Qué cuenta como empleo? '
            'Formato: ✅/⚠️/❌ + 2-3 oraciones. Si captura la idea central, es correcta. Español.',
            f'Pregunta: «El 95% de nuestros graduados consigue empleo en 6 meses.» ¿Qué preguntarías?\nRespuesta: {quiz_3.value}', max_tokens=300)
        print(fb)
        pts = 2 if fb.startswith('✅') else 1 if fb.startswith('⚠️') else 0
        add_score('quiz', pts, 2)
        show_score()

btn_q3 = widgets.Button(description='Evaluar', button_style='primary')
btn_q3.on_click(eval_q3)
display(quiz_3, btn_q3, out_q3)

In [ ]:
#@title Quiz — Pregunta 4
display(HTML("<div style='background:#37474f;color:white;padding:10px;border-radius:6px;'>"
             f"<b>Pregunta 4:</b> Clasifica: «Si automatizamos estos 3 reportes, el departamento ahorra 120 horas/mes.»</div>"))

quiz_4 = widgets.RadioButtons(options=['Descriptivo', 'Exploratorio', 'Inferencial', 'Predictivo', 'Prescriptivo'], value=None, layout=widgets.Layout(width='100%'))
out_q4 = widgets.Output()

def eval_q4(btn):
    btn.disabled = True
    with out_q4:
        clear_output()
        if quiz_4.value == 'Prescriptivo':
            print('✅ ¡Correcto! Prescriptivo: recomienda acción + cuantifica resultado.')
            add_score('quiz', 2, 2)
        else:
            print(f'❌ Elegiste: {quiz_4.value}. Prescriptivo: recomienda acción + cuantifica resultado.')
            add_score('quiz', 0, 2)
        show_score()

btn_q4 = widgets.Button(description='Verificar', button_style='primary')
btn_q4.on_click(eval_q4)
display(quiz_4, btn_q4, out_q4)

In [ ]:
#@title Quiz — Pregunta 5
display(HTML("<div style='background:#37474f;color:white;padding:10px;border-radius:6px;'>"
             f"<b>Pregunta 5:</b> Un docente dice: «En mi experiencia, los estudiantes de la mañana rinden más.» ¿Es análisis de datos?</div>"))

quiz_5 = widgets.Textarea(placeholder='Tu respuesta...', layout=widgets.Layout(width='100%', height='70px'))
out_q5 = widgets.Output()

def eval_q5(btn):
    btn.disabled = True
    with out_q5:
        clear_output()
        if not quiz_5.value.strip(): print('⚠️ Sin respuesta'); add_score('quiz', 0, 2); show_score(); return
        fb = ask_llm(
            'Evalúa esta respuesta de BA. Correcta/esperada: No. Es intuición anecdótica. Necesita: datos sistemáticos, medición controlada, control de variables. '
            'Formato: ✅/⚠️/❌ + 2-3 oraciones. Si captura la idea central, es correcta. Español.',
            f'Pregunta: Un docente dice: «En mi experiencia, los estudiantes de la mañana rinden más.» ¿Es análisis de datos?\nRespuesta: {quiz_5.value}', max_tokens=300)
        print(fb)
        pts = 2 if fb.startswith('✅') else 1 if fb.startswith('⚠️') else 0
        add_score('quiz', pts, 2)
        show_score()

btn_q5 = widgets.Button(description='Evaluar', button_style='primary')
btn_q5.on_click(eval_q5)
display(quiz_5, btn_q5, out_q5)

---
## 🏆 Resultados Finales
> **¿Por qué resultados por módulo?** Para que veas tu perfil de competencias, no solo una nota global. Un 70% general puede significar que dominas sesgos pero fallas en clasificación de variables. El desglose te muestra exactamente dónde invertir tu tiempo de estudio.


### 🐍 Python en esta celda

**Iteración sobre diccionario** — `for key, name in modulo_names.items():` recorre cada par clave-nombre. `.items()` devuelve tuplas `(clave, valor)`.

**Operador ternario con cálculo** — `pct = m['total'] / m['max'] * 100 if m['max'] > 0 else 0` calcula el porcentaje solo si `max > 0` (evita dividir entre cero).

**`.append()`** — Agrega un elemento al final de una lista. `data_m.append((name, pct))` va construyendo la lista de resultados por módulo.

In [ ]:
#@title Ver mis resultados finales
if score['max'] == 0:
    print('⚠️ No has completado ninguna actividad todavía.')
else:
    modulo_names = {'recap': 'Recap Relámpago', 'caso_desercion': 'Caso Deserción',
                    'sesgos': 'Detective de Sesgos', 'datos_reales': 'Datos Reales Venezuela',
                    'lab01': 'Lab 01 — Tu Proyecto', 'quiz': 'Quiz Integrador'}
    data_m = []
    for key, name in modulo_names.items():
        if key in score['modulo']:
            m = score['modulo'][key]
            pct = m['total'] / m['max'] * 100 if m['max'] > 0 else 0
            data_m.append({'Módulo': name, 'Puntaje': f"{m['total']}/{m['max']}", '%': f'{pct:.0f}%'})
    if data_m:
        display(HTML(pd.DataFrame(data_m).to_html(index=False)))

    pct_total = score['total'] / score['max'] * 100
    color = '#4CAF50' if pct_total >= 70 else '#FF9800' if pct_total >= 50 else '#f44336'
    display(HTML(f"<div style='background:{color};color:white;padding:20px;border-radius:12px;text-align:center;"
                 f"margin:20px 0;font-size:20px;'><b>PUNTAJE TOTAL: {score['total']}/{score['max']} ({pct_total:.0f}%)</b></div>"))

> **¿Por qué un radar?** Porque muestra fortalezas y debilidades de forma simultánea y visual. Es la misma herramienta que se usa en evaluaciones 360° en organizaciones — y la estás experimentando aplicada a tu propio aprendizaje.

### 🐍 Python en esta celda

**`go.Scatterpolar(r=vals, theta=cats)`** — Crea un gráfico de radar (polar). `r` son los valores (distancia al centro), `theta` son las categorías (etiquetas alrededor del círculo).

**Truco del cierre** — `cats + [cats[0]]` y `vals + [vals[0]]` repiten el primer valor al final para que la línea del radar se cierre visualmente. Sin esto, quedaría un espacio abierto entre el último y el primer eje.

In [ ]:
#@title Gráfico radar de competencias
modulo_names = {'recap': 'Recap', 'caso_desercion': 'Caso Deserción',
                'sesgos': 'Sesgos', 'datos_reales': 'Datos Reales',
                'lab01': 'Lab 01', 'quiz': 'Quiz'}
data_m = []
for key, name in modulo_names.items():
    if key in score['modulo']:
        m = score['modulo'][key]
        pct = m['total'] / m['max'] * 100 if m['max'] > 0 else 0
        data_m.append((name, pct))

if len(data_m) >= 3:
    cats = [d[0] for d in data_m] + [data_m[0][0]]
    vals = [d[1] for d in data_m] + [data_m[0][1]]
    fig = go.Figure(data=go.Scatterpolar(r=vals, theta=cats, fill='toself',
                    fillcolor='rgba(33,150,243,0.3)', line=dict(color='#2196F3', width=2)))
    fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
                      title='Perfil de competencias — Módulo 01', height=400, template='plotly_white')
    fig.show()
else:
    print('⚠️ Completa al menos 3 módulos para ver el radar.')

> **¿Por qué feedback final de IA?** Para cerrar con una síntesis personalizada. La IA cruza tus puntajes por módulo y genera recomendaciones específicas — el equivalente a que un tutor revise tu trabajo y te diga exactamente qué estudiar para la siguiente clase.

### 🐍 Python en esta celda

**Construcción de prompt dinámico** — El ciclo construye `modulos_str` con los resultados de cada módulo. Ese string se inserta en el prompt de la IA para que genere feedback basado en *tus* datos reales, no en respuestas genéricas.

**`display(Markdown(texto))`** — Renderiza texto con formato Markdown (negritas, listas, cursivas) directamente en el notebook. Más legible que `print()` para respuestas largas.

In [ ]:
#@title Feedback final personalizado de la IA
modulo_names = {'recap': 'Recap', 'caso_desercion': 'Caso Deserción',
                'sesgos': 'Sesgos', 'datos_reales': 'Datos Reales',
                'lab01': 'Lab 01', 'quiz': 'Quiz'}
modulos_str = ''
for key, name in modulo_names.items():
    if key in score['modulo']:
        m = score['modulo'][key]
        pct = m['total'] / m['max'] * 100 if m['max'] > 0 else 0
        modulos_str += f'- {name}: {pct:.0f}%\n'

pct_total = score['total'] / score['max'] * 100 if score['max'] > 0 else 0

feedback_final = ask_llm(
    'Eres mentor de Business Analytics para docentes universitarios.\n'
    'Basándote en los resultados, da feedback personalizado:\n'
    '1. Fortaleza principal (1 oración)\n2. Área de mejora (1 oración)\n'
    '3. Recomendación para Módulo 2: Data Wrangling (1 oración)\nMotivador pero honesto. Español.',
    f'Resultados:\n{modulos_str}Total: {pct_total:.0f}%', max_tokens=400)

display(HTML(f"<div style='background:#f5f5f5;padding:15px;border-radius:8px;border-left:4px solid #2196F3;'>"
             f"<b>🤖 Feedback personalizado:</b><br><br>{feedback_final}</div>"))

display(HTML("""
<div style='background:#263238;color:white;padding:20px;border-radius:10px;margin-top:20px;text-align:center;'>
    <i>"Pensar con datos no es saber estadística.<br>
    Es hacerse mejores preguntas, reconocer lo que no sabes,<br>
    y ser honesto con las limitaciones de lo que mides."</i>
    <br><br>
    <b>Entregable:</b> Lab 01 con feedback incorporado → 13 de abril de 2026<br>
    <b>Módulo 2:</b> Data Wrangling Accesible — Python y SQL 🐍
</div>"""))